In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import torch
import time
from IPython.display import Video, display

# =========================
# CONFIGURACIÓN
# =========================

MODEL_PROFE = "best.pt"
MODEL_NUEVO = "runs_ball/Ball_Padel_LeMar_yolov8s/weights/best.pt"

# Cambia este video por el que quieras probar
VIDEO_PATH = "data/videos/Partido_1/Punto_1.mp4"

OUTPUT_DIR = Path("comparacion_modelos_ball/videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = OUTPUT_DIR / "comparacion_profe_vs_nuevo.mp4"

IMG_SIZE = 960
CONF = 0.05   # bajo para ver si el modelo del profe detecta algo
MAX_FRAMES = None  # pon 300 si quieres probar solo 300 frames

# =========================
# DEVICE
# =========================

if torch.cuda.is_available():
    device = 0
    print(f"✅ CUDA disponible: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    print("✅ MPS disponible")
else:
    device = "cpu"
    print("⚠️ Usando CPU")

# =========================
# CARGAR MODELOS
# =========================

model_profe = YOLO(MODEL_PROFE)
model_nuevo = YOLO(MODEL_NUEVO)

print("Clases modelo profe:", model_profe.names)
print("Clases modelo nuevo:", model_nuevo.names)

# =========================
# FUNCIONES
# =========================

def dibujar_detecciones(frame, result, titulo):
    frame = frame.copy()

    boxes = result.boxes

    # Título
    cv2.rectangle(frame, (0, 0), (frame.shape[1], 45), (0, 0, 0), -1)
    cv2.putText(
        frame,
        titulo,
        (15, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

    if boxes is None or len(boxes) == 0:
        cv2.putText(
            frame,
            "Sin detecciones",
            (15, 85),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2,
            cv2.LINE_AA
        )
        return frame, 0, 0.0

    detecciones = 0
    conf_total = 0.0

    for box in boxes:
        conf = float(box.conf[0])
        cls = int(box.cls[0])

        x1, y1, x2, y2 = map(int, box.xyxy[0])

        detecciones += 1
        conf_total += conf

        label = f"Ball {conf:.2f}"

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        cv2.putText(
            frame,
            label,
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

    conf_promedio = conf_total / detecciones if detecciones > 0 else 0.0

    return frame, detecciones, conf_promedio


def redimensionar(frame, width=640):
    h, w = frame.shape[:2]
    scale = width / w
    new_h = int(h * scale)
    return cv2.resize(frame, (width, new_h))

# =========================
# ABRIR VIDEO
# =========================

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {VIDEO_PATH}")
print(f"FPS original: {fps}")
print(f"Frames totales: {total_frames}")

# Leer primer frame para definir tamaño
ret, frame = cap.read()

if not ret:
    raise ValueError("No se pudo leer el primer frame del video.")

frame_resized = redimensionar(frame, width=640)
h_out, w_single = frame_resized.shape[:2]
w_out = w_single * 2

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, fps, (w_out, h_out))

# Reiniciar video
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# =========================
# PROCESAR VIDEO
# =========================

frame_idx = 0

stats = {
    "profe_detecciones": 0,
    "nuevo_detecciones": 0,
    "profe_frames_con_deteccion": 0,
    "nuevo_frames_con_deteccion": 0,
    "profe_conf_sum": 0.0,
    "nuevo_conf_sum": 0.0,
}

inicio = time.time()

while True:
    ret, frame = cap.read()

    if not ret:
        break

    if MAX_FRAMES is not None and frame_idx >= MAX_FRAMES:
        break

    # Predicción modelo profe
    result_profe = model_profe.predict(
        source=frame,
        imgsz=IMG_SIZE,
        conf=CONF,
        device=device,
        verbose=False
    )[0]

    # Predicción modelo nuevo
    result_nuevo = model_nuevo.predict(
        source=frame,
        imgsz=IMG_SIZE,
        conf=CONF,
        device=device,
        verbose=False
    )[0]

    frame_profe, det_profe, conf_profe = dibujar_detecciones(
        frame,
        result_profe,
        "Modelo profe"
    )

    frame_nuevo, det_nuevo, conf_nuevo = dibujar_detecciones(
        frame,
        result_nuevo,
        "Modelo nuevo Ball_Padel_LeMar"
    )

    # Estadísticas
    stats["profe_detecciones"] += det_profe
    stats["nuevo_detecciones"] += det_nuevo

    if det_profe > 0:
        stats["profe_frames_con_deteccion"] += 1
        stats["profe_conf_sum"] += conf_profe

    if det_nuevo > 0:
        stats["nuevo_frames_con_deteccion"] += 1
        stats["nuevo_conf_sum"] += conf_nuevo

    # Redimensionar para comparar lado a lado
    frame_profe = redimensionar(frame_profe, width=640)
    frame_nuevo = redimensionar(frame_nuevo, width=640)

    comparacion = cv2.hconcat([frame_profe, frame_nuevo])

    writer.write(comparacion)

    frame_idx += 1

    if frame_idx % 50 == 0:
        print(f"Procesados {frame_idx} frames...")

cap.release()
writer.release()

tiempo_total = time.time() - inicio

# =========================
# RESUMEN
# =========================

frames_procesados = frame_idx

profe_conf_prom = (
    stats["profe_conf_sum"] / stats["profe_frames_con_deteccion"]
    if stats["profe_frames_con_deteccion"] > 0 else 0
)

nuevo_conf_prom = (
    stats["nuevo_conf_sum"] / stats["nuevo_frames_con_deteccion"]
    if stats["nuevo_frames_con_deteccion"] > 0 else 0
)

print("\n" + "=" * 70)
print("📊 COMPARACIÓN EN VIDEO")
print("=" * 70)

print(f"Frames procesados: {frames_procesados}")
print(f"Tiempo total: {tiempo_total:.2f} s")
print(f"FPS procesamiento aprox: {frames_procesados / tiempo_total:.2f}")

print("\nModelo profe:")
print(f"- Detecciones totales: {stats['profe_detecciones']}")
print(f"- Frames con detección: {stats['profe_frames_con_deteccion']}")
print(f"- % frames con detección: {(stats['profe_frames_con_deteccion'] / frames_procesados) * 100:.2f}%")
print(f"- Confianza promedio: {profe_conf_prom:.3f}")

print("\nModelo nuevo:")
print(f"- Detecciones totales: {stats['nuevo_detecciones']}")
print(f"- Frames con detección: {stats['nuevo_frames_con_deteccion']}")
print(f"- % frames con detección: {(stats['nuevo_frames_con_deteccion'] / frames_procesados) * 100:.2f}%")
print(f"- Confianza promedio: {nuevo_conf_prom:.3f}")

print("\n✅ Video comparativo guardado en:")
print(OUTPUT_VIDEO)

# Mostrar video en Jupyter
display(Video(str(OUTPUT_VIDEO), embed=True))

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import torch
import time
from IPython.display import Video, display
import numpy as np
from sklearn.metrics import roc_auc_score

# =========================
# CONFIGURACIÓN
# =========================

MODEL_PROFE = "best.pt"
MODEL_NUEVO = "runs_ball/Ball_Padel_LeMar_yolov8s/weights/best.pt"
VIDEO_PATH = "data/videos/Partido_1/Punto_1.mp4"

OUTPUT_DIR = Path("comparacion_modelos_ball/videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = OUTPUT_DIR / "comparacion_profe_vs_nuevo.mp4"

IMG_SIZE = 960
CONF = 0.05
MAX_FRAMES = None

IOU_THRESH = 0.5

# =========================
# DEVICE
# =========================

if torch.cuda.is_available():
    device = 0
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# =========================
# MODELOS
# =========================

model_profe = YOLO(MODEL_PROFE)
model_nuevo = YOLO(MODEL_NUEVO)

print("Clases profe:", model_profe.names)
print("Clases nuevo:", model_nuevo.names)

# =========================
# IOU
# =========================

def iou(box1, box2):
    xi1 = max(box1[0], box2[0])
    yi1 = max(box1[1], box2[1])
    xi2 = min(box1[2], box2[2])
    yi2 = min(box1[3], box2[3])

    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)

    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = area1 + area2 - inter

    return inter / union if union > 0 else 0

def match_detections(pred_boxes, gt_boxes):
    tp = 0
    fp = 0
    fn = len(gt_boxes)
    used = []

    for pb in pred_boxes:
        matched = False

        for i, gb in enumerate(gt_boxes):
            if i in used:
                continue

            if iou(pb, gb) >= IOU_THRESH:
                tp += 1
                used.append(i)
                fn -= 1
                matched = True
                break

        if not matched:
            fp += 1

    return tp, fp, fn

def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# =========================
# DIBUJAR
# =========================

def dibujar(frame, result, titulo):
    frame = frame.copy()

    cv2.rectangle(frame, (0, 0), (frame.shape[1], 45), (0, 0, 0), -1)
    cv2.putText(frame, titulo, (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

    boxes = result.boxes

    if boxes is None or len(boxes) == 0:
        cv2.putText(frame, "Sin detecciones", (15, 85),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        return frame, [], 0, 0.0

    pred_boxes = []
    confs = []

    for b in boxes:
        conf = float(b.conf[0])
        x1, y1, x2, y2 = map(int, b.xyxy[0])

        pred_boxes.append([x1, y1, x2, y2])
        confs.append(conf)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"{conf:.2f}", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    return frame, pred_boxes, len(pred_boxes), np.mean(confs)

def resize(frame, width=640):
    h, w = frame.shape[:2]
    scale = width / w
    return cv2.resize(frame, (width, int(h * scale)))

# =========================
# VIDEO
# =========================

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError("No se pudo abrir video")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

ret, frame = cap.read()
frame = resize(frame, 640)
h, w = frame.shape[:2]

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w * 2, h)
)

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# =========================
# ESTADÍSTICAS
# =========================

stats = {
    "profe_det": 0,
    "nuevo_det": 0,
    "profe_frames": 0,
    "nuevo_frames": 0,
    "profe_conf": 0,
    "nuevo_conf": 0,
}

# ROC-AUC (opcional)
y_true = []
y_scores = []

tp_all = fp_all = fn_all = 0

start = time.time()
frame_idx = 0

# =========================
# LOOP
# =========================

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if MAX_FRAMES and frame_idx >= MAX_FRAMES:
        break

    # predicciones
    r1 = model_profe.predict(frame, imgsz=IMG_SIZE, conf=CONF, device=device, verbose=False)[0]
    r2 = model_nuevo.predict(frame, imgsz=IMG_SIZE, conf=CONF, device=device, verbose=False)[0]

    f1, boxes1, n1, c1 = dibujar(frame, r1, "Modelo PROFE")
    f2, boxes2, n2, c2 = dibujar(frame, r2, "Modelo NUEVO")

    # stats simples
    stats["profe_det"] += n1
    stats["nuevo_det"] += n2

    if n1 > 0:
        stats["profe_frames"] += 1
        stats["profe_conf"] += c1

    if n2 > 0:
        stats["nuevo_frames"] += 1
        stats["nuevo_conf"] += c2

    # métricas supervisadas (solo si tienes GT)
    # AQUÍ GT VACÍO POR DEFECTO (si tienes labels lo conectamos)
    gt_boxes = []

    tp, fp, fn = match_detections(boxes1, gt_boxes)
    tp_all += tp
    fp_all += fp
    fn_all += fn

    # ROC (solo demo si GT existe)
    for c in [c1] * len(boxes1):
        y_scores.append(c)
        y_true.append(1 if gt_boxes else 0)

    f1 = cv2.hconcat([resize(f1), resize(f2)])
    writer.write(f1)

    frame_idx += 1

    if frame_idx % 50 == 0:
        print("Frames:", frame_idx)

cap.release()
writer.release()

# =========================
# RESULTADOS
# =========================

time_total = time.time() - start

precision, recall, f1_score = compute_metrics(tp_all, fp_all, fn_all)

profe_conf_avg = stats["profe_conf"] / stats["profe_frames"] if stats["profe_frames"] else 0
nuevo_conf_avg = stats["nuevo_conf"] / stats["nuevo_frames"] if stats["nuevo_frames"] else 0

roc_auc = None
if len(set(y_true)) > 1:
    roc_auc = roc_auc_score(y_true, y_scores)

print("\n" + "="*60)
print("📊 RESULTADOS")
print("="*60)

print(f"Frames: {frame_idx}")
print(f"Tiempo: {time_total:.2f}s")
print(f"FPS: {frame_idx/time_total:.2f}")

print("\n📌 MODELO PROFE")
print(f"- Detecciones: {stats['profe_det']}")
print(f"- % frames con detección: {100*stats['profe_frames']/frame_idx:.2f}%")
print(f"- Confianza media: {profe_conf_avg:.3f}")

print("\n📌 MODELO NUEVO")
print(f"- Detecciones: {stats['nuevo_det']}")
print(f"- % frames con detección: {100*stats['nuevo_frames']/frame_idx:.2f}%")
print(f"- Confianza media: {nuevo_conf_avg:.3f}")

print("\n📊 MÉTRICAS (si hay GT)")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1_score:.3f}")

print(f"ROC-AUC: {roc_auc if roc_auc else 'NO DISPONIBLE (sin GT válido)'}")

print("\n💾 Video guardado en:", OUTPUT_VIDEO)

display(Video(str(OUTPUT_VIDEO), embed=True))

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import torch
import time
import os
import numpy as np
from IPython.display import Video, display
from sklearn.metrics import roc_auc_score

# =========================
# CONFIG
# =========================

MODEL_PROFE = "best.pt"
MODEL_NUEVO = "runs_ball/Ball_Padel_LeMar_yolov8s/weights/best.pt"

VIDEO_PATH = "data/videos/Partido_1/Punto_1.mp4"

LABELS_PATH = "data/datasets/Ball_Padel_LeMar_split/val/labels"

OUTPUT_DIR = Path("comparacion_modelos_ball/videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = OUTPUT_DIR / "comparacion_profe_vs_nuevo.mp4"

IMG_SIZE = 960
CONF = 0.05
MAX_FRAMES = None
IOU_THRESH = 0.5

# =========================
# DEVICE
# =========================

if torch.cuda.is_available():
    device = 0
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# =========================
# MODELOS
# =========================

model_profe = YOLO(MODEL_PROFE)
model_nuevo = YOLO(MODEL_NUEVO)

print("Clases profe:", model_profe.names)
print("Clases nuevo:", model_nuevo.names)

# =========================
# IOU
# =========================

def iou(box1, box2):
    xi1 = max(box1[0], box2[0])
    yi1 = max(box1[1], box2[1])
    xi2 = min(box1[2], box2[2])
    yi2 = min(box1[3], box2[3])

    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union = area1 + area2 - inter

    return inter / union if union > 0 else 0


def match_detections(pred_boxes, gt_boxes):
    tp = 0
    fp = 0
    fn = len(gt_boxes)
    used = []

    for pb in pred_boxes:
        matched = False

        for i, gb in enumerate(gt_boxes):
            if i in used:
                continue

            if iou(pb, gb) >= IOU_THRESH:
                tp += 1
                fn -= 1
                used.append(i)
                matched = True
                break

        if not matched:
            fp += 1

    return tp, fp, fn


def compute_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# =========================
# YOLO LABEL LOADER
# =========================

def load_yolo_labels(label_path, w, h):
    boxes = []

    if not os.path.exists(label_path):
        return boxes

    with open(label_path, "r") as f:
        lines = f.readlines()

    for l in lines:
        parts = l.strip().split()
        if len(parts) < 5:
            continue

        cls, xc, yc, bw, bh = map(float, parts)

        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w
        y2 = (yc + bh / 2) * h

        boxes.append([x1, y1, x2, y2])

    return boxes

# =========================
# DRAW
# =========================

def draw(frame, result, title):
    frame = frame.copy()

    cv2.rectangle(frame, (0, 0), (frame.shape[1], 45), (0, 0, 0), -1)
    cv2.putText(frame, title, (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

    boxes = result.boxes

    pred_boxes = []
    confs = []

    if boxes is None or len(boxes) == 0:
        return frame, [], 0, 0

    for b in boxes:
        conf = float(b.conf[0])
        x1, y1, x2, y2 = map(int, b.xyxy[0])

        pred_boxes.append([x1, y1, x2, y2])
        confs.append(conf)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"{conf:.2f}", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    return frame, pred_boxes, len(pred_boxes), np.mean(confs)

# =========================
# VIDEO
# =========================

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)

ret, frame = cap.read()
h, w = frame.shape[:2]

def resize(img, width=640):
    scale = width / img.shape[1]
    return cv2.resize(img, (width, int(img.shape[0] * scale)))

frame_r = resize(frame, 640)
h, w = frame_r.shape[:2]

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w * 2, h)
)

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# =========================
# STATS
# =========================

stats = {
    "profe_det": 0,
    "nuevo_det": 0,
    "profe_frames": 0,
    "nuevo_frames": 0,
    "profe_conf": 0,
    "nuevo_conf": 0,
}

tp_all = fp_all = fn_all = 0

start = time.time()
frame_idx = 0

# =========================
# LOOP
# =========================

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if MAX_FRAMES and frame_idx >= MAX_FRAMES:
        break

    # GT LABEL PATH
    label_file = os.path.join(LABELS_PATH, f"frame_{frame_idx:06d}.txt")
    gt_boxes = load_yolo_labels(label_file, w, h)

    # predictions
    r1 = model_profe.predict(frame, imgsz=IMG_SIZE, conf=CONF, device=device, verbose=False)[0]
    r2 = model_nuevo.predict(frame, imgsz=IMG_SIZE, conf=CONF, device=device, verbose=False)[0]

    f1, pb1, n1, c1 = draw(frame, r1, "PROFE")
    f2, pb2, n2, c2 = draw(frame, r2, "NUEVO")

    stats["profe_det"] += n1
    stats["nuevo_det"] += n2

    if n1 > 0:
        stats["profe_frames"] += 1
        stats["profe_conf"] += c1

    if n2 > 0:
        stats["nuevo_frames"] += 1
        stats["nuevo_conf"] += c2

    # METRICS (PROFE MODEL)
    tp, fp, fn = match_detections(pb1, gt_boxes)
    tp_all += tp
    fp_all += fp
    fn_all += fn

    combined = cv2.hconcat([resize(f1), resize(f2)])
    writer.write(combined)

    frame_idx += 1

    if frame_idx % 50 == 0:
        print("Frames:", frame_idx)

cap.release()
writer.release()

# =========================
# FINAL METRICS
# =========================

time_total = time.time() - start

precision, recall, f1 = compute_metrics(tp_all, fp_all, fn_all)

profe_conf_avg = stats["profe_conf"] / stats["profe_frames"] if stats["profe_frames"] else 0
nuevo_conf_avg = stats["nuevo_conf"] / stats["nuevo_frames"] if stats["nuevo_frames"] else 0

print("\n" + "="*60)
print("📊 RESULTADOS FINALES")
print("="*60)

print(f"Frames: {frame_idx}")
print(f"Tiempo: {time_total:.2f}s")
print(f"FPS: {frame_idx/time_total:.2f}")

print("\n📌 MODELO PROFE")
print(f"- Detecciones: {stats['profe_det']}")
print(f"- % frames con detección: {100*stats['profe_frames']/frame_idx:.2f}%")
print(f"- Confianza media: {profe_conf_avg:.3f}")

print("\n📌 MODELO NUEVO")
print(f"- Detecciones: {stats['nuevo_det']}")
print(f"- % frames con detección: {100*stats['nuevo_frames']/frame_idx:.2f}%")
print(f"- Confianza media: {nuevo_conf_avg:.3f}")

print("\n📊 MÉTRICAS REALES (PROFE vs GT)")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")

print("\n💾 Video:", OUTPUT_VIDEO)

display(Video(str(OUTPUT_VIDEO), embed=True))

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import torch
import time
import os
import numpy as np
from glob import glob
from IPython.display import Video, display

# =========================
# CONFIG
# =========================

MODEL_PROFE = "best.pt"
MODEL_NUEVO = "runs_ball/Ball_Padel_LeMar_yolov8s/weights/best.pt"

DATASET_PATH = "data/datasets/Ball_Padel_LeMar_split/val"
IMAGES_PATH = f"{DATASET_PATH}/images"
LABELS_PATH = f"{DATASET_PATH}/labels"

OUTPUT_DIR = Path("comparacion_modelos_ball/videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = OUTPUT_DIR / "VS_profe_vs_nuevo.mp4"

IMG_SIZE = 960
CONF = 0.05
IOU_THRESH = 0.5
MAX_FRAMES = None

# =========================
# DEVICE
# =========================

device = 0 if torch.cuda.is_available() else "cpu"

# =========================
# MODELOS
# =========================

model_profe = YOLO(MODEL_PROFE)
model_nuevo = YOLO(MODEL_NUEVO)

print("Clases profe:", model_profe.names)
print("Clases nuevo:", model_nuevo.names)

# =========================
# LOAD DATASET
# =========================

image_paths = sorted(glob(IMAGES_PATH + "/*"))

def get_label_path(img_path):
    name = os.path.splitext(os.path.basename(img_path))[0] + ".txt"
    return os.path.join(LABELS_PATH, name)

# =========================
# YOLO LABELS
# =========================

def load_yolo_labels(label_path, w, h):
    boxes = []

    if not os.path.exists(label_path):
        return boxes

    with open(label_path, "r") as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            _, xc, yc, bw, bh = map(float, parts)

            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h
            x2 = (xc + bw / 2) * w
            y2 = (yc + bh / 2) * h

            boxes.append([x1, y1, x2, y2])

    return boxes

# =========================
# IOU
# =========================

def iou(a, b):
    xi1 = max(a[0], b[0])
    yi1 = max(a[1], b[1])
    xi2 = min(a[2], b[2])
    yi2 = min(a[3], b[3])

    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)

    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])

    union = area_a + area_b - inter

    return inter / union if union > 0 else 0

# =========================
# MATCHING
# =========================

def match(pred, gt):
    tp = 0
    fp = 0
    fn = len(gt)
    used = []

    for p in pred:
        found = False

        for i, g in enumerate(gt):
            if i in used:
                continue

            if iou(p, g) >= IOU_THRESH:
                tp += 1
                fn -= 1
                used.append(i)
                found = True
                break

        if not found:
            fp += 1

    return tp, fp, fn

# =========================
# STATS
# =========================

stats = {
    "profe_det": 0,
    "nuevo_det": 0,
    "profe_frames": 0,
    "nuevo_frames": 0,
    "profe_conf": 0,
    "nuevo_conf": 0,
}

tp_p = fp_p = fn_p = 0
tp_n = fp_n = fn_n = 0

# =========================
# VIDEO WRITER
# =========================

cap0 = cv2.imread(image_paths[0])
h, w = cap0.shape[:2]

def resize(img, width=640):
    scale = width / img.shape[1]
    return cv2.resize(img, (width, int(img.shape[0] * scale)))

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    25,
    (1280, 720)
)

# =========================
# LOOP
# =========================

start = time.time()

for idx, img_path in enumerate(image_paths):

    if MAX_FRAMES and idx >= MAX_FRAMES:
        break

    frame = cv2.imread(img_path)
    h, w = frame.shape[:2]

    label_path = get_label_path(img_path)
    gt_boxes = load_yolo_labels(label_path, w, h)

    # predictions
    r1 = model_profe.predict(frame, imgsz=IMG_SIZE, conf=CONF, device=device, verbose=False)[0]
    r2 = model_nuevo.predict(frame, imgsz=IMG_SIZE, conf=CONF, device=device, verbose=False)[0]

    pred_p = []
    pred_n = []

    conf_p = []
    conf_n = []

    # PROFE
    if r1.boxes is not None:
        for b in r1.boxes:
            x1, y1, x2, y2 = map(int, b.xyxy[0])
            pred_p.append([x1, y1, x2, y2])
            conf_p.append(float(b.conf[0]))

    # NUEVO
    if r2.boxes is not None:
        for b in r2.boxes:
            x1, y1, x2, y2 = map(int, b.xyxy[0])
            pred_n.append([x1, y1, x2, y2])
            conf_n.append(float(b.conf[0]))

    # STATS
    stats["profe_det"] += len(pred_p)
    stats["nuevo_det"] += len(pred_n)

    if len(pred_p) > 0:
        stats["profe_frames"] += 1
        stats["profe_conf"] += np.mean(conf_p)

    if len(pred_n) > 0:
        stats["nuevo_frames"] += 1
        stats["nuevo_conf"] += np.mean(conf_n)

    # METRICS REAL VS GT
    tp, fp, fn = match(pred_p, gt_boxes)
    tp_p += tp
    fp_p += fp
    fn_p += fn

    tp, fp, fn = match(pred_n, gt_boxes)
    tp_n += tp
    fp_n += fp
    fn_n += fn

    # DRAW SIMPLE VS
    cv2.putText(frame, "PROFE vs NUEVO", (30, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)

    cv2.putText(frame, f"P:{len(pred_p)} N:{len(pred_n)} GT:{len(gt_boxes)}",
                (30, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)

    frame = resize(frame, 1280)

    writer.write(frame)

    if idx % 50 == 0:
        print("Frames:", idx)

# =========================
# FINAL METRICS
# =========================

def metrics(tp, fp, fn):
    p = tp / (tp + fp) if (tp+fp) else 0
    r = tp / (tp + fn) if (tp+fn) else 0
    f1 = 2*p*r/(p+r) if (p+r) else 0
    return p, r, f1

p_p, r_p, f1_p = metrics(tp_p, fp_p, fn_p)
p_n, r_n, f1_n = metrics(tp_n, fp_n, fn_n)

time_total = time.time() - start

print("\n" + "="*70)
print("📊 VS FINAL PROFE vs NUEVO")
print("="*70)

print(f"Frames: {len(image_paths)}")
print(f"Tiempo: {time_total:.2f}s")

print("\n📌 PROFE")
print(f"Detecciones: {stats['profe_det']}")
print(f"Precision: {p_p:.3f} Recall: {r_p:.3f} F1: {f1_p:.3f}")

print("\n📌 NUEVO")
print(f"Detecciones: {stats['nuevo_det']}")
print(f"Precision: {p_n:.3f} Recall: {r_n:.3f} F1: {f1_n:.3f}")

print("\n📌 CONFIANZA")
print(f"Profe: {stats['profe_conf']/max(1,stats['profe_frames']):.3f}")
print(f"Nuevo: {stats['nuevo_conf']/max(1,stats['nuevo_frames']):.3f}")

print("\n💾 Video:", OUTPUT_VIDEO)

writer.release()

display(Video(str(OUTPUT_VIDEO), embed=True))

In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
import time
from glob import glob
from pathlib import Path
from IPython.display import Video, display

# =========================
# CONFIG
# =========================

MODEL_PROFE = "best.pt"
MODEL_NUEVO = "runs_ball/Ball_Padel_LeMar_yolov8s/weights/best.pt"

DATASET = "data/datasets/Ball_Padel_LeMar_split/val"
IMAGES_DIR = f"{DATASET}/images"
LABELS_DIR = f"{DATASET}/labels"

OUTPUT_DIR = Path("comparacion_modelos_ball/videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = OUTPUT_DIR / "FINAL_VS_PROFE_NUEVO.mp4"

IMG_SIZE = 960
CONF = 0.05
IOU_THRESH = 0.5
MAX_FRAMES = None

# =========================
# MODELOS
# =========================

device = 0

model_profe = YOLO(MODEL_PROFE)
model_nuevo = YOLO(MODEL_NUEVO)

print("Clases profe:", model_profe.names)
print("Clases nuevo:", model_nuevo.names)

# =========================
# DATASET
# =========================

image_paths = sorted(glob(IMAGES_DIR + "/*"))

def get_label_path(img_path):
    name = os.path.splitext(os.path.basename(img_path))[0] + ".txt"
    return os.path.join(LABELS_DIR, name)

# =========================
# UTILIDADES
# =========================

def load_labels(label_path, w, h):
    boxes = []
    if not os.path.exists(label_path):
        return boxes

    with open(label_path, "r") as f:
        lines = f.readlines()

    for l in lines:
        parts = l.strip().split()
        if len(parts) < 5:
            continue

        _, xc, yc, bw, bh = map(float, parts)

        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w
        y2 = (yc + bh / 2) * h

        boxes.append([x1, y1, x2, y2])

    return boxes


def iou(a, b):
    xi1 = max(a[0], b[0])
    yi1 = max(a[1], b[1])
    xi2 = min(a[2], b[2])
    yi2 = min(a[3], b[3])

    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)

    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])

    union = area_a + area_b - inter

    return inter / union if union > 0 else 0


def match(pred, gt):
    tp, fp = 0, 0
    fn = len(gt)
    used = []

    for p in pred:
        found = False
        for i, g in enumerate(gt):
            if i in used:
                continue
            if iou(p, g) >= IOU_THRESH:
                tp += 1
                fn -= 1
                used.append(i)
                found = True
                break
        if not found:
            fp += 1

    return tp, fp, fn


def metrics(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0
    r = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * p * r / (p + r) if (p + r) else 0
    return p, r, f1

# =========================
# VIDEO WRITER
# =========================

first = cv2.imread(image_paths[0])
h0, w0 = first.shape[:2]

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    25,
    (1280, 720)
)

# =========================
# STATS
# =========================

tp_p = fp_p = fn_p = 0
tp_n = fp_n = fn_n = 0

stats = {
    "profe_det": 0,
    "nuevo_det": 0,
    "profe_frames": 0,
    "nuevo_frames": 0,
    "profe_conf": 0,
    "nuevo_conf": 0,
}

start = time.time()

# =========================
# LOOP PRINCIPAL
# =========================

for idx, img_path in enumerate(image_paths):

    if MAX_FRAMES and idx >= MAX_FRAMES:
        break

    frame = cv2.imread(img_path)

    # 🔥 FIX CRÍTICO
    if frame is None:
        print(f"⚠️ Imagen corrupta o no encontrada: {img_path}")
        continue

    h, w = frame.shape[:2]

    label_path = get_label_path(img_path)
    gt_boxes = load_labels(label_path, w, h)

    # =========================
    # PREDICCIONES
    # =========================

    r1 = model_profe.predict(frame, imgsz=IMG_SIZE, conf=CONF, device=device, verbose=False)[0]
    r2 = model_nuevo.predict(frame, imgsz=IMG_SIZE, conf=CONF, device=device, verbose=False)[0]

    pred_p, pred_n = [], []
    conf_p, conf_n = [], []

    if r1.boxes:
        for b in r1.boxes:
            x1, y1, x2, y2 = map(int, b.xyxy[0])
            pred_p.append([x1, y1, x2, y2])
            conf_p.append(float(b.conf[0]))

    if r2.boxes:
        for b in r2.boxes:
            x1, y1, x2, y2 = map(int, b.xyxy[0])
            pred_n.append([x1, y1, x2, y2])
            conf_n.append(float(b.conf[0]))

    # =========================
    # STATS
    # =========================

    stats["profe_det"] += len(pred_p)
    stats["nuevo_det"] += len(pred_n)

    if pred_p:
        stats["profe_frames"] += 1
        stats["profe_conf"] += np.mean(conf_p)

    if pred_n:
        stats["nuevo_frames"] += 1
        stats["nuevo_conf"] += np.mean(conf_n)

    # =========================
    # MATCHING REAL
    # =========================

    tp, fp, fn = match(pred_p, gt_boxes)
    tp_p += tp
    fp_p += fp
    fn_p += fn

    tp, fp, fn = match(pred_n, gt_boxes)
    tp_n += tp
    fp_n += fp
    fn_n += fn

    # =========================
    # VISUAL
    # =========================

    cv2.putText(frame, "PROFE vs NUEVO", (30, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)

    cv2.putText(frame,
                f"GT:{len(gt_boxes)} P:{len(pred_p)} N:{len(pred_n)}",
                (30, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255,255,255),
                2)

    frame = cv2.resize(frame, (1280, 720))
    writer.write(frame)

    if idx % 50 == 0:
        print(f"Frames procesados: {idx}")

# =========================
# RESULTADOS
# =========================

writer.release()

time_total = time.time() - start

def fmt(tp, fp, fn):
    p, r, f1 = metrics(tp, fp, fn)
    return p, r, f1

p_p, r_p, f1_p = fmt(tp_p, fp_p, fn_p)
p_n, r_n, f1_n = fmt(tp_n, fp_n, fn_n)

print("\n" + "="*70)
print("📊 RESULTADO FINAL - PROFE vs NUEVO")
print("="*70)

print(f"Frames: {len(image_paths)}")
print(f"Tiempo: {time_total:.2f}s")
print(f"FPS: {len(image_paths)/time_total:.2f}")

print("\n📌 PROFE")
print(f"Detecciones: {stats['profe_det']}")
print(f"Precision: {p_p:.3f} | Recall: {r_p:.3f} | F1: {f1_p:.3f}")

print("\n📌 NUEVO")
print(f"Detecciones: {stats['nuevo_det']}")
print(f"Precision: {p_n:.3f} | Recall: {r_n:.3f} | F1: {f1_n:.3f}")

print("\n📌 CONFIANZA")
print(f"Profe: {stats['profe_conf']/max(1,stats['profe_frames']):.3f}")
print(f"Nuevo: {stats['nuevo_conf']/max(1,stats['nuevo_frames']):.3f}")

print("\n💾 Video:", OUTPUT_VIDEO)

display(Video(str(OUTPUT_VIDEO), embed=True))

In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
import time
from glob import glob
from pathlib import Path
from IPython.display import Video, display

# =========================
# CONFIG
# =========================

MODEL_PROFE = "best.pt"
MODEL_NUEVO = "runs_ball/Ball_Padel_LeMar_yolov8s/weights/best.pt"

DATASET = "data/datasets/Ball_Padel_LeMar_split/val"
IMAGES_DIR = f"{DATASET}/images"
LABELS_DIR = f"{DATASET}/labels"

OUTPUT_DIR = Path("comparacion_modelos_ball/videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = OUTPUT_DIR / "FINAL_VS_PROFE_NUEVO.mp4"

IMG_SIZE = 960
CONF = 0.05
IOU_THRESH = 0.5
MAX_FRAMES = None

# =========================
# MODELOS
# =========================

device = 0

model_profe = YOLO(MODEL_PROFE)
model_nuevo = YOLO(MODEL_NUEVO)

print("Clases profe:", model_profe.names)
print("Clases nuevo:", model_nuevo.names)

# =========================
# DATASET (FIX IMPORTANTE)
# =========================

valid_ext = {".jpg", ".jpeg", ".png"}

image_paths = sorted([
    str(p) for p in Path(IMAGES_DIR).rglob("*")
    if p.suffix.lower() in valid_ext
])

def get_label_path(img_path):
    name = os.path.splitext(os.path.basename(img_path))[0] + ".txt"
    return os.path.join(LABELS_DIR, name)

# =========================
# UTILIDADES
# =========================

def load_labels(label_path, w, h):
    boxes = []
    if not os.path.exists(label_path):
        return boxes

    with open(label_path, "r") as f:
        lines = f.readlines()

    for l in lines:
        parts = l.strip().split()
        if len(parts) < 5:
            continue

        _, xc, yc, bw, bh = map(float, parts)

        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w
        y2 = (yc + bh / 2) * h

        boxes.append([x1, y1, x2, y2])

    return boxes


def iou(a, b):
    xi1 = max(a[0], b[0])
    yi1 = max(a[1], b[1])
    xi2 = min(a[2], b[2])
    yi2 = min(a[3], b[3])

    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)

    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])

    union = area_a + area_b - inter

    return inter / union if union > 0 else 0


def match(pred, gt):
    tp, fp = 0, 0
    fn = len(gt)
    used = []

    for p in pred:
        found = False
        for i, g in enumerate(gt):
            if i in used:
                continue
            if iou(p, g) >= IOU_THRESH:
                tp += 1
                fn -= 1
                used.append(i)
                found = True
                break
        if not found:
            fp += 1

    return tp, fp, fn


def metrics(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0
    r = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * p * r / (p + r) if (p + r) else 0
    return p, r, f1

# =========================
# VIDEO WRITER
# =========================

first = cv2.imread(image_paths[0])
h0, w0 = first.shape[:2]

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    25,
    (1280, 720)
)

# =========================
# STATS
# =========================

tp_p = fp_p = fn_p = 0
tp_n = fp_n = fn_n = 0

stats = {
    "profe_det": 0,
    "nuevo_det": 0,
    "profe_frames": 0,
    "nuevo_frames": 0,
    "profe_conf": 0,
    "nuevo_conf": 0,
}

start = time.time()

# =========================
# LOOP PRINCIPAL
# =========================

for idx, img_path in enumerate(image_paths):

    if MAX_FRAMES and idx >= MAX_FRAMES:
        break

    frame = cv2.imread(img_path)

    # FIX: ignorar archivos inválidos
    if frame is None:
        print(f"⚠️ Imagen corrupta o no válida: {img_path}")
        continue

    h, w = frame.shape[:2]

    label_path = get_label_path(img_path)
    gt_boxes = load_labels(label_path, w, h)

    # =========================
    # PREDICCIONES
    # =========================

    r1 = model_profe.predict(frame, imgsz=IMG_SIZE, conf=CONF, device=device, verbose=False)[0]
    r2 = model_nuevo.predict(frame, imgsz=IMG_SIZE, conf=CONF, device=device, verbose=False)[0]

    pred_p, pred_n = [], []
    conf_p, conf_n = [], []

    if r1.boxes:
        for b in r1.boxes:
            x1, y1, x2, y2 = map(int, b.xyxy[0])
            pred_p.append([x1, y1, x2, y2])
            conf_p.append(float(b.conf[0]))

    if r2.boxes:
        for b in r2.boxes:
            x1, y1, x2, y2 = map(int, b.xyxy[0])
            pred_n.append([x1, y1, x2, y2])
            conf_n.append(float(b.conf[0]))

    # =========================
    # STATS
    # =========================

    stats["profe_det"] += len(pred_p)
    stats["nuevo_det"] += len(pred_n)

    if pred_p:
        stats["profe_frames"] += 1
        stats["profe_conf"] += np.mean(conf_p)

    if pred_n:
        stats["nuevo_frames"] += 1
        stats["nuevo_conf"] += np.mean(conf_n)

    # =========================
    # MATCHING
    # =========================

    tp, fp, fn = match(pred_p, gt_boxes)
    tp_p += tp
    fp_p += fp
    fn_p += fn

    tp, fp, fn = match(pred_n, gt_boxes)
    tp_n += tp
    fp_n += fp
    fn_n += fn

    # =========================
    # VISUAL
    # =========================

    cv2.putText(frame, "PROFE vs NUEVO", (30, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)

    cv2.putText(frame,
                f"GT:{len(gt_boxes)} P:{len(pred_p)} N:{len(pred_n)}",
                (30, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255,255,255),
                2)

    frame = cv2.resize(frame, (1280, 720))
    writer.write(frame)

    if idx % 50 == 0:
        print(f"Frames procesados: {idx}")

# =========================
# RESULTADOS
# =========================

writer.release()

time_total = time.time() - start

def fmt(tp, fp, fn):
    p, r, f1 = metrics(tp, fp, fn)
    return p, r, f1

p_p, r_p, f1_p = fmt(tp_p, fp_p, fn_p)
p_n, r_n, f1_n = fmt(tp_n, fp_n, fn_n)

print("\n" + "="*70)
print("📊 RESULTADO FINAL - PROFE vs NUEVO")
print("="*70)

print(f"Frames: {len(image_paths)}")
print(f"Tiempo: {time_total:.2f}s")
print(f"FPS: {len(image_paths)/time_total:.2f}")

print("\n📌 PROFE")
print(f"Detecciones: {stats['profe_det']}")
print(f"Precision: {p_p:.3f} | Recall: {r_p:.3f} | F1: {f1_p:.3f}")

print("\n📌 NUEVO")
print(f"Detecciones: {stats['nuevo_det']}")
print(f"Precision: {p_n:.3f} | Recall: {r_n:.3f} | F1: {f1_n:.3f}")

print("\n📌 CONFIANZA")
print(f"Profe: {stats['profe_conf']/max(1,stats['profe_frames']):.3f}")
print(f"Nuevo: {stats['nuevo_conf']/max(1,stats['nuevo_frames']):.3f}")

print("\n💾 Video:", OUTPUT_VIDEO)

display(Video(str(OUTPUT_VIDEO), embed=True))

In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
import time
from pathlib import Path
from IPython.display import Video, display

# =========================
# CONFIG
# =========================

MODEL_PROFE = "best.pt"
MODEL_NUEVO = "runs_ball/Ball_Padel_LeMar_yolov8s/weights/best.pt"

DATASET = "data/datasets/Ball_Padel_LeMar_split/val"
IMAGES_DIR = f"{DATASET}/images"
LABELS_DIR = f"{DATASET}/labels"

OUTPUT_DIR = Path("comparacion_modelos_ball/videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = OUTPUT_DIR / "SIDE_BY_SIDE_PROFE_VS_NUEVO.mp4"

IMG_SIZE = 960
CONF = 0.05
MAX_FRAMES = None

device = 0

# =========================
# MODELOS
# =========================

model_profe = YOLO(MODEL_PROFE)
model_nuevo = YOLO(MODEL_NUEVO)

print("Clases profe:", model_profe.names)
print("Clases nuevo:", model_nuevo.names)

# =========================
# DATASET
# =========================

valid_ext = {".jpg", ".jpeg", ".png"}

image_paths = sorted([
    str(p) for p in Path(IMAGES_DIR).rglob("*")
    if p.suffix.lower() in valid_ext
])

def get_best_box(result):
    """Devuelve SOLO la detección con mayor confianza"""
    if result.boxes is None or len(result.boxes) == 0:
        return None, None

    confs = result.boxes.conf.cpu().numpy()
    best_i = int(np.argmax(confs))

    b = result.boxes[best_i]
    x1, y1, x2, y2 = map(int, b.xyxy[0])
    conf = float(b.conf[0])

    return [x1, y1, x2, y2], conf

# =========================
# VIDEO WRITER
# =========================

writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    1,
    (1280, 720)
)

# =========================
# FUNCION DIBUJO
# =========================

def draw_box(img, box, color, label):
    if box is None:
        return

    x1, y1, x2, y2 = box

    # centro (cuadrito pequeño)
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
    size = 8

    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    cv2.rectangle(img,
                  (cx - size, cy - size),
                  (cx + size, cy + size),
                  color, -1)

    cv2.putText(img, label, (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6, color, 2)

# =========================
# LOOP PRINCIPAL
# =========================

start = time.time()

for idx, img_path in enumerate(image_paths):

    if MAX_FRAMES and idx >= MAX_FRAMES:
        break

    frame = cv2.imread(img_path)
    if frame is None:
        continue

    # =========================
    # PREDICCIONES
    # =========================
    r1 = model_profe.predict(frame, imgsz=IMG_SIZE, conf=CONF, device=device, verbose=False)[0]
    r2 = model_nuevo.predict(frame, imgsz=IMG_SIZE, conf=CONF, device=device, verbose=False)[0]

    box_p, conf_p = get_best_box(r1)
    box_n, conf_n = get_best_box(r2)

    # =========================
    # COPIAS PARA CADA MODELO
    # =========================
    frame_p = frame.copy()
    frame_n = frame.copy()

    draw_box(
        frame_p,
        box_p,
        (0, 255, 255),
        f"PROFE {conf_p:.2f}" if box_p else "PROFE NO DET"
    )

    draw_box(
        frame_n,
        box_n,
        (0, 255, 0),
        f"NUEVO {conf_n:.2f}" if box_n else "NUEVO NO DET"
    )

    # =========================
    # RESIZE + CONCAT
    # =========================
    frame_p = cv2.resize(frame_p, (640, 720))
    frame_n = cv2.resize(frame_n, (640, 720))

    combined = np.hstack((frame_p, frame_n))

    cv2.putText(combined,
                "PROFE (izq) vs NUEVO (der)",
                (30, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (255, 255, 255),
                2)

    writer.write(combined)

    if idx % 50 == 0:
        print(f"Procesados: {idx}")

# =========================
# FINAL
# =========================

writer.release()

print("\n✅ Video guardado en:", OUTPUT_VIDEO)

display(Video(str(OUTPUT_VIDEO), embed=True))

In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
import time
import json
from pathlib import Path
from IPython.display import Video, display, HTML
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, confusion_matrix,
    precision_recall_curve, roc_curve, average_precision_score
)
import warnings
warnings.filterwarnings("ignore")

# =========================
# CONFIG
# =========================
MODEL_PROFE = "best.pt"
MODEL_NUEVO = "runs_ball/Ball_Padel_LeMar_yolov8s/weights/best.pt"
DATASET     = "data/datasets/Ball_Padel_LeMar_split/val"
IMAGES_DIR  = f"{DATASET}/images"
LABELS_DIR  = f"{DATASET}/labels"
OUTPUT_DIR  = Path("comparacion_modelos_ball")
VIDEO_DIR   = OUTPUT_DIR / "videos"
PLOT_DIR    = OUTPUT_DIR / "plots"
for d in [VIDEO_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO = VIDEO_DIR / "SIDE_BY_SIDE_PROFE_VS_NUEVO.mp4"
OUTPUT_HTML  = OUTPUT_DIR / "reporte_analitico.html"

IMG_SIZE   = 960
CONF       = 0.05
IOU_THRESH = 0.5          # IoU mínimo para contar como TP
MAX_FRAMES = None
device     = 0

# ─────────────────────────────────────────────
# COLORES por modelo
# ─────────────────────────────────────────────
COLOR_PROFE = (0, 255, 255)   # cyan  (BGR)
COLOR_NUEVO = (0, 255, 0)     # verde (BGR)
HEX_PROFE   = "#00FFFF"
HEX_NUEVO   = "#00FF88"
HEX_BG      = "#0d1117"

# =========================
# MODELOS
# =========================
model_profe = YOLO(MODEL_PROFE)
model_nuevo = YOLO(MODEL_NUEVO)
print("Clases profe:", model_profe.names)
print("Clases nuevo:", model_nuevo.names)

# =========================
# DATASET
# =========================
valid_ext   = {".jpg", ".jpeg", ".png"}
image_paths = sorted([
    str(p) for p in Path(IMAGES_DIR).rglob("*")
    if p.suffix.lower() in valid_ext
])
print(f"Total imágenes: {len(image_paths)}")

# =========================
# HELPERS
# =========================

def get_best_box(result):
    """Devuelve SOLO la detección con mayor confianza."""
    if result.boxes is None or len(result.boxes) == 0:
        return None, None, 0.0
    confs   = result.boxes.conf.cpu().numpy()
    best_i  = int(np.argmax(confs))
    b       = result.boxes[best_i]
    x1, y1, x2, y2 = map(int, b.xyxy[0])
    conf    = float(b.conf[0])
    return [x1, y1, x2, y2], conf, conf          # box, conf, raw_score


def load_gt_box(label_path, img_w, img_h):
    """Carga la primera bbox del label YOLO (normalizado → píxeles)."""
    if not os.path.exists(label_path):
        return None
    with open(label_path) as f:
        lines = [l.strip() for l in f if l.strip()]
    if not lines:
        return None
    parts = lines[0].split()
    if len(parts) < 5:
        return None
    _, cx, cy, bw, bh = map(float, parts[:5])
    x1 = int((cx - bw / 2) * img_w)
    y1 = int((cy - bh / 2) * img_h)
    x2 = int((cx + bw / 2) * img_w)
    y2 = int((cy + bh / 2) * img_h)
    return [x1, y1, x2, y2]


def iou(boxA, boxB):
    """Intersection over Union entre dos boxes [x1,y1,x2,y2]."""
    if boxA is None or boxB is None:
        return 0.0
    xA = max(boxA[0], boxB[0]);  yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]);  yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0


def draw_box(img, box, color, label):
    if box is None:
        return
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
    size   = 8
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    cv2.rectangle(img, (cx-size, cy-size), (cx+size, cy+size), color, -1)
    cv2.putText(img, label, (x1, max(y1-10, 15)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)


def draw_gt_box(img, box):
    if box is None:
        return
    cv2.rectangle(img, (box[0], box[1]), (box[2], box[3]), (255, 100, 0), 2)
    cv2.putText(img, "GT", (box[0], max(box[1]-10, 15)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 100, 0), 1)

# =========================
# ACUMULADORES
# =========================
stats = {
    "profe": {"y_true": [], "y_pred": [], "y_score": [],
              "tp": 0, "fp": 0, "fn": 0, "tn": 0,
              "iou_list": [], "conf_list": [], "inf_ms": []},
    "nuevo": {"y_true": [], "y_pred": [], "y_score": [],
              "tp": 0, "fp": 0, "fn": 0, "tn": 0,
              "iou_list": [], "conf_list": [], "inf_ms": []},
}

# =========================
# VIDEO WRITER
# =========================
writer = cv2.VideoWriter(
    str(OUTPUT_VIDEO),
    cv2.VideoWriter_fourcc(*"mp4v"),
    1,
    (1280, 720)
)

# =========================
# LOOP PRINCIPAL
# =========================
t_start = time.time()

for idx, img_path in enumerate(image_paths):
    if MAX_FRAMES and idx >= MAX_FRAMES:
        break

    frame = cv2.imread(img_path)
    if frame is None:
        continue
    h, w = frame.shape[:2]

    # Ground-truth
    stem       = Path(img_path).stem
    label_path = Path(LABELS_DIR) / (stem + ".txt")
    gt_box     = load_gt_box(str(label_path), w, h)
    has_gt     = gt_box is not None

    # ── Inferencia ──────────────────────────────────────────
    for key, model in [("profe", model_profe), ("nuevo", model_nuevo)]:
        t0  = time.perf_counter()
        res = model.predict(frame, imgsz=IMG_SIZE, conf=CONF,
                            device=device, verbose=False)[0]
        inf_ms = (time.perf_counter() - t0) * 1000
        stats[key]["inf_ms"].append(inf_ms)

        box, conf, score = get_best_box(res)
        detected = box is not None

        # Score para ROC (usa conf si detectó, 0 si no)
        y_score = score if detected else 0.0
        stats[key]["y_score"].append(y_score)
        stats[key]["y_true"].append(1 if has_gt else 0)
        stats[key]["conf_list"].append(conf if detected else 0.0)

        # TP / FP / FN / TN + IoU
        if has_gt and detected:
            iou_val = iou(gt_box, box)
            stats[key]["iou_list"].append(iou_val)
            if iou_val >= IOU_THRESH:
                stats[key]["tp"] += 1
                stats[key]["y_pred"].append(1)
            else:
                stats[key]["fp"] += 1
                stats[key]["fn"] += 1
                stats[key]["y_pred"].append(0)
        elif has_gt and not detected:
            stats[key]["fn"] += 1
            stats[key]["y_pred"].append(0)
        elif not has_gt and detected:
            stats[key]["fp"] += 1
            stats[key]["y_pred"].append(1)
        else:  # not has_gt and not detected
            stats[key]["tn"] += 1
            stats[key]["y_pred"].append(0)

    # ── Video frame ─────────────────────────────────────────
    box_p, conf_p, _ = get_best_box(
        model_profe.predict(frame, imgsz=IMG_SIZE, conf=CONF,
                            device=device, verbose=False)[0])
    box_n, conf_n, _ = get_best_box(
        model_nuevo.predict(frame, imgsz=IMG_SIZE, conf=CONF,
                            device=device, verbose=False)[0])

    frame_p = frame.copy()
    frame_n = frame.copy()

    draw_gt_box(frame_p, gt_box)
    draw_gt_box(frame_n, gt_box)

    draw_box(frame_p, box_p, COLOR_PROFE,
             f"PROFE {conf_p:.2f}" if box_p else "PROFE NO DET")
    draw_box(frame_n, box_n, COLOR_NUEVO,
             f"NUEVO {conf_n:.2f}" if box_n else "NUEVO NO DET")

    frame_p = cv2.resize(frame_p, (640, 720))
    frame_n = cv2.resize(frame_n, (640, 720))
    combined = np.hstack((frame_p, frame_n))

    cv2.putText(combined, "PROFE (izq) vs NUEVO (der)",
                (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
    writer.write(combined)

    if idx % 50 == 0:
        elapsed = time.time() - t_start
        print(f"  [{idx}/{len(image_paths)}] {elapsed:.1f}s")

writer.release()
print(f"\n✅ Video: {OUTPUT_VIDEO}")

# ================================================================
# CÁLCULO DE MÉTRICAS
# ================================================================
def compute_metrics(s):
    y_true  = np.array(s["y_true"])
    y_pred  = np.array(s["y_pred"])
    y_score = np.array(s["y_score"])

    tp, fp, fn, tn = s["tp"], s["fp"], s["fn"], s["tn"]

    prec   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1     = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    acc    = (tp + tn) / (tp + tn + fp + fn) if (tp+tn+fp+fn) > 0 else 0.0

    # ROC-AUC (solo si hay ambas clases)
    try:
        roc_auc = roc_auc_score(y_true, y_score)
    except Exception:
        roc_auc = float("nan")

    try:
        avg_prec = average_precision_score(y_true, y_score)
    except Exception:
        avg_prec = float("nan")

    mean_iou   = float(np.mean(s["iou_list"])) if s["iou_list"] else 0.0
    mean_conf  = float(np.mean([c for c in s["conf_list"] if c > 0])) if any(c > 0 for c in s["conf_list"]) else 0.0
    mean_inf   = float(np.mean(s["inf_ms"])) if s["inf_ms"] else 0.0

    return dict(
        accuracy=acc, precision=prec, recall=rec, f1=f1,
        roc_auc=roc_auc, avg_precision=avg_prec,
        mean_iou=mean_iou, mean_conf=mean_conf,
        mean_inf_ms=mean_inf,
        tp=tp, fp=fp, fn=fn, tn=tn,
        y_true=y_true, y_score=y_score,
        iou_list=s["iou_list"],
    )

m_p = compute_metrics(stats["profe"])
m_n = compute_metrics(stats["nuevo"])

print("\n══════════════════════════════════════════")
print("   MÉTRICAS FINALES")
print("══════════════════════════════════════════")
print(f"{'Métrica':<22} {'PROFE':>10} {'NUEVO':>10}")
print("─" * 46)
for k in ["accuracy","precision","recall","f1","roc_auc","avg_precision","mean_iou","mean_conf","mean_inf_ms"]:
    vp = m_p[k]; vn = m_n[k]
    print(f"  {k:<20} {vp:>10.4f} {vn:>10.4f}")
print(f"  {'TP':<20} {m_p['tp']:>10} {m_n['tp']:>10}")
print(f"  {'FP':<20} {m_p['fp']:>10} {m_n['fp']:>10}")
print(f"  {'FN':<20} {m_p['fn']:>10} {m_n['fn']:>10}")
print(f"  {'TN':<20} {m_p['tn']:>10} {m_n['tn']:>10}")

# ================================================================
# PLOTS
# ================================================================
DARK   = "#0d1117"
GREY   = "#161b22"
TEXT   = "#e6edf3"
ACCENT = "#58a6ff"
plt.rcParams.update({
    "figure.facecolor": DARK, "axes.facecolor": GREY,
    "axes.edgecolor": "#30363d", "axes.labelcolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT,
    "text.color": TEXT, "grid.color": "#21262d",
    "font.family": "monospace",
})

# ── 1. Radar / Bar – métricas principales ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(DARK)
metrics_keys = ["accuracy","precision","recall","f1","roc_auc","avg_precision","mean_iou"]
labels       = ["Accuracy","Precision","Recall","F1","ROC-AUC","mAP","Mean IoU"]
vp = [m_p[k] for k in metrics_keys]
vn = [m_n[k] for k in metrics_keys]

x  = np.arange(len(labels))
w  = 0.35
ax = axes[0]
ax.bar(x - w/2, vp, w, label="PROFE", color=HEX_PROFE, alpha=0.9)
ax.bar(x + w/2, vn, w, label="NUEVO", color=HEX_NUEVO, alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
ax.set_ylim(0, 1.1); ax.set_title("Comparación de Métricas", fontsize=11, color=TEXT)
ax.legend(facecolor=GREY, edgecolor="#30363d")
ax.grid(axis="y", alpha=0.4)

# ── 2. Matriz de confusión lado a lado ────────────────────────
ax2 = axes[1]
ax2.axis("off")
table_data = [
    ["",      "Pred 0",     "Pred 1"    ],
    ["Act 0", f"TN={m_p['tn']}", f"FP={m_p['fp']}" ],
    ["Act 1", f"FN={m_p['fn']}", f"TP={m_p['tp']}" ],
]
table2 = [
    ["",      "Pred 0",     "Pred 1"    ],
    ["Act 0", f"TN={m_n['tn']}", f"FP={m_n['fp']}" ],
    ["Act 1", f"FN={m_n['fn']}", f"TP={m_n['tp']}" ],
]
y_base = 0.85
for title, tdata, color in [
        ("PROFE – Confusion Matrix", table_data, HEX_PROFE),
        ("NUEVO – Confusion Matrix", table2,     HEX_NUEVO)]:
    ax2.text(0.02 if "PROFE" in title else 0.52, y_base+0.08,
             title, transform=ax2.transAxes, color=color, fontsize=9, fontweight="bold")
    for r, row in enumerate(tdata):
        for c, val in enumerate(row):
            xp = (0.02 if "PROFE" in title else 0.52) + c * 0.14
            yp = y_base - r * 0.22
            bg = "#1f2937" if r == 0 or c == 0 else (
                 "#134e2a" if (r == 2 and c == 2) else
                 "#7f1d1d" if (r == 1 and c == 2 or r == 2 and c == 1) else "#1f2937")
            ax2.text(xp, yp, val, transform=ax2.transAxes,
                     ha="center", va="center", fontsize=8, color=TEXT,
                     bbox=dict(facecolor=bg, edgecolor="#30363d", boxstyle="round,pad=0.3"))

plt.tight_layout()
bar_path = PLOT_DIR / "metricas_barras.png"
plt.savefig(bar_path, dpi=130, bbox_inches="tight", facecolor=DARK)
plt.close()
print(f"  Saved: {bar_path}")

# ── 3. ROC Curve ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(DARK); ax.set_facecolor(GREY)
for (label, m, color) in [("PROFE", m_p, HEX_PROFE), ("NUEVO", m_n, HEX_NUEVO)]:
    try:
        fpr, tpr, _ = roc_curve(m["y_true"], m["y_score"])
        ax.plot(fpr, tpr, color=color, lw=2,
                label=f"{label}  AUC={m['roc_auc']:.3f}")
        ax.fill_between(fpr, tpr, alpha=0.08, color=color)
    except Exception:
        pass
ax.plot([0,1],[0,1],"--", color="#30363d")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve", color=TEXT)   
ax.legend(facecolor=GREY, edgecolor="#30363d"); ax.grid(alpha=0.3)
roc_path = PLOT_DIR / "roc_curve.png"
plt.savefig(roc_path, dpi=130, bbox_inches="tight", facecolor=DARK)
plt.close()
print(f"  Saved: {roc_path}")

# ── 4. Precision-Recall Curve ────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(DARK); ax.set_facecolor(GREY)
for (label, m, color) in [("PROFE", m_p, HEX_PROFE), ("NUEVO", m_n, HEX_NUEVO)]:
    try:
        p, r, _ = precision_recall_curve(m["y_true"], m["y_score"])
        ax.plot(r, p, color=color, lw=2,
                label=f"{label}  AP={m['avg_precision']:.3f}")
        ax.fill_between(r, p, alpha=0.08, color=color)
    except Exception:
        pass
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve", color=TEXT)
ax.legend(facecolor=GREY, edgecolor="#30363d"); ax.grid(alpha=0.3)
pr_path = PLOT_DIR / "pr_curve.png"
plt.savefig(pr_path, dpi=130, bbox_inches="tight", facecolor=DARK)
plt.close()
print(f"  Saved: {pr_path}")

# ── 5. IoU Distribution ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(DARK); ax.set_facecolor(GREY)
if m_p["iou_list"]:
    ax.hist(m_p["iou_list"], bins=30, color=HEX_PROFE, alpha=0.7,
            label=f"PROFE  μ={m_p['mean_iou']:.3f}")
if m_n["iou_list"]:
    ax.hist(m_n["iou_list"], bins=30, color=HEX_NUEVO, alpha=0.7,
            label=f"NUEVO  μ={m_n['mean_iou']:.3f}")
ax.axvline(IOU_THRESH, color="white", linestyle="--", alpha=0.6, label=f"IoU≥{IOU_THRESH}")
ax.set_xlabel("IoU"); ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de IoU (detecciones sobre GT)", color=TEXT)
ax.legend(facecolor=GREY, edgecolor="#30363d"); ax.grid(alpha=0.3)
iou_path = PLOT_DIR / "iou_distribution.png"
plt.savefig(iou_path, dpi=130, bbox_inches="tight", facecolor=DARK)
plt.close()
print(f"  Saved: {iou_path}")

# ── 6. Inference time ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(DARK); ax.set_facecolor(GREY)
ax.hist(stats["profe"]["inf_ms"], bins=40, color=HEX_PROFE, alpha=0.7,
        label=f"PROFE  μ={m_p['mean_inf_ms']:.1f}ms")
ax.hist(stats["nuevo"]["inf_ms"], bins=40, color=HEX_NUEVO, alpha=0.7,
        label=f"NUEVO  μ={m_n['mean_inf_ms']:.1f}ms")
ax.set_xlabel("Tiempo inferencia (ms)"); ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de Tiempo de Inferencia", color=TEXT)
ax.legend(facecolor=GREY, edgecolor="#30363d"); ax.grid(alpha=0.3)
inf_path = PLOT_DIR / "inference_time.png"
plt.savefig(inf_path, dpi=130, bbox_inches="tight", facecolor=DARK)
plt.close()
print(f"  Saved: {inf_path}")

# ================================================================
# REPORTE HTML
# ================================================================
def pct(v):
    return f"{v*100:.2f}%" if not np.isnan(v) else "N/A"

def ms(v):
    return f"{v:.1f} ms"

winner = lambda vp, vn: ("🏆", "—") if vp > vn else ("—", "🏆") if vn > vp else ("—", "—")

rows = [
    ("Accuracy",         m_p["accuracy"],        m_n["accuracy"],        True),
    ("Precision",        m_p["precision"],        m_n["precision"],       True),
    ("Recall",           m_p["recall"],           m_n["recall"],          True),
    ("F1-Score",         m_p["f1"],               m_n["f1"],              True),
    ("ROC-AUC",          m_p["roc_auc"],          m_n["roc_auc"],         True),
    ("Avg Precision (mAP)", m_p["avg_precision"], m_n["avg_precision"],   True),
    ("Mean IoU",         m_p["mean_iou"],         m_n["mean_iou"],        True),
    ("Mean Confidence",  m_p["mean_conf"],        m_n["mean_conf"],       True),
    ("Inf. Time (ms)",   m_p["mean_inf_ms"],      m_n["mean_inf_ms"],     False),  # lower is better
]

table_rows_html = ""
for name, vp, vn, higher_better in rows:
    if higher_better:
        wp, wn = winner(vp, vn)
    else:
        wp, wn = winner(-vp, -vn)
    fmt = pct if name not in ("Mean Confidence","Inf. Time (ms)","Mean IoU") else (
          ms  if "ms" in name else pct)
    vp_s = f"{vp*100:.2f}%" if "ms" not in name and name != "Mean Confidence" else (
           f"{vp:.3f}" if name == "Mean Confidence" else f"{vp:.1f} ms")
    vn_s = f"{vn*100:.2f}%" if "ms" not in name and name != "Mean Confidence" else (
           f"{vn:.3f}" if name == "Mean Confidence" else f"{vn:.1f} ms")
    table_rows_html += f"""
        <tr>
          <td>{name}</td>
          <td class="val cyan">{vp_s} {wp}</td>
          <td class="val green">{vn_s} {wn}</td>
        </tr>"""

cm_rows_html = ""
for label, m, color in [("PROFE", m_p, "cyan"), ("NUEVO", m_n, "green")]:
    cm_rows_html += f"""
    <div class="cm-block">
      <h3 class="{color}">{label} – Confusion Matrix</h3>
      <table class="cm-table">
        <tr><th></th><th>Pred 0</th><th>Pred 1</th></tr>
        <tr><td>Act 0</td>
            <td class="tn">TN<br>{m['tn']}</td>
            <td class="fp">FP<br>{m['fp']}</td></tr>
        <tr><td>Act 1</td>
            <td class="fn">FN<br>{m['fn']}</td>
            <td class="tp">TP<br>{m['tp']}</td></tr>
      </table>
    </div>"""

html_content = f"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<title>Reporte Analítico – Comparación de Modelos</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600;700&family=Space+Grotesk:wght@400;600;700&display=swap');
  *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
  :root {{
    --bg: #0d1117; --surface: #161b22; --border: #30363d;
    --text: #e6edf3; --muted: #8b949e;
    --cyan: #00ffff; --green: #00ff88; --blue: #58a6ff;
    --red: #ff6b6b; --yellow: #ffd700;
  }}
  body {{ background: var(--bg); color: var(--text); font-family: 'Space Grotesk', sans-serif; padding: 2rem; }}
  h1 {{ font-family: 'JetBrains Mono', monospace; font-size: 1.6rem; color: var(--cyan); margin-bottom: 0.3rem; }}
  .subtitle {{ color: var(--muted); font-size: 0.85rem; margin-bottom: 2rem; }}
  .section {{ margin-bottom: 2.5rem; }}
  h2 {{ font-family: 'JetBrains Mono', monospace; font-size: 1rem; color: var(--blue); border-bottom: 1px solid var(--border); padding-bottom: 0.5rem; margin-bottom: 1rem; }}
  h3 {{ font-size: 0.9rem; margin-bottom: 0.5rem; }}
  .cyan {{ color: var(--cyan); }}
  .green {{ color: var(--green); }}
  /* Metrics table */
  .metrics-table {{ width: 100%; border-collapse: collapse; font-family: 'JetBrains Mono', monospace; font-size: 0.82rem; }}
  .metrics-table th {{ background: var(--surface); color: var(--muted); padding: 0.5rem 1rem; text-align: left; border: 1px solid var(--border); }}
  .metrics-table td {{ padding: 0.5rem 1rem; border: 1px solid var(--border); }}
  .metrics-table tr:nth-child(even) td {{ background: #0f1419; }}
  .val {{ text-align: right; font-weight: 600; font-size: 0.9rem; }}
  .metrics-table td.cyan {{ color: var(--cyan); }}
  .metrics-table td.green {{ color: var(--green); }}
  /* CM */
  .cm-wrap {{ display: flex; gap: 2rem; flex-wrap: wrap; }}
  .cm-block {{ background: var(--surface); border: 1px solid var(--border); border-radius: 8px; padding: 1rem 1.5rem; }}
  .cm-table {{ border-collapse: collapse; font-family: 'JetBrains Mono', monospace; font-size: 0.85rem; margin-top: 0.5rem; }}
  .cm-table th, .cm-table td {{ border: 1px solid var(--border); padding: 0.5rem 1rem; text-align: center; }}
  .cm-table th {{ background: #21262d; color: var(--muted); }}
  .tp {{ background: #0d2818; color: #4ade80; font-weight: 700; }}
  .tn {{ background: #1a2540; color: var(--blue); font-weight: 700; }}
  .fp {{ background: #3b1d1d; color: var(--red); font-weight: 700; }}
  .fn {{ background: #3b2600; color: var(--yellow); font-weight: 700; }}
  /* Plots */
  .plots-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(420px, 1fr)); gap: 1rem; }}
  .plot-card {{ background: var(--surface); border: 1px solid var(--border); border-radius: 8px; overflow: hidden; }}
  .plot-card img {{ width: 100%; display: block; }}
  .plot-card p {{ padding: 0.4rem 0.8rem; font-size: 0.75rem; color: var(--muted); }}
  /* Summary cards */
  .cards {{ display: flex; gap: 1rem; flex-wrap: wrap; margin-bottom: 1.5rem; }}
  .card {{ background: var(--surface); border: 1px solid var(--border); border-radius: 8px; padding: 1rem 1.5rem; min-width: 160px; }}
  .card .label {{ font-size: 0.7rem; color: var(--muted); text-transform: uppercase; letter-spacing: 0.08em; }}
  .card .value {{ font-size: 1.4rem; font-weight: 700; font-family: 'JetBrains Mono', monospace; margin-top: 0.2rem; }}
  footer {{ margin-top: 3rem; font-size: 0.75rem; color: var(--muted); border-top: 1px solid var(--border); padding-top: 1rem; }}
</style>
</head>
<body>

<h1>🎾 Model Analytics Report</h1>
<p class="subtitle">PROFE vs NUEVO · Ball Padel Detection · IoU threshold = {IOU_THRESH} · Generated automatically</p>

<div class="section">
  <h2>01 / Summary Scorecards</h2>
  <div class="cards">
    <div class="card"><div class="label">PROFE F1</div><div class="value cyan">{m_p['f1']*100:.1f}%</div></div>
    <div class="card"><div class="label">NUEVO F1</div><div class="value green">{m_n['f1']*100:.1f}%</div></div>
    <div class="card"><div class="label">PROFE ROC-AUC</div><div class="value cyan">{m_p['roc_auc']:.3f}</div></div>
    <div class="card"><div class="label">NUEVO ROC-AUC</div><div class="value green">{m_n['roc_auc']:.3f}</div></div>
    <div class="card"><div class="label">PROFE Mean IoU</div><div class="value cyan">{m_p['mean_iou']:.3f}</div></div>
    <div class="card"><div class="label">NUEVO Mean IoU</div><div class="value green">{m_n['mean_iou']:.3f}</div></div>
    <div class="card"><div class="label">PROFE Inf.</div><div class="value cyan">{m_p['mean_inf_ms']:.1f}ms</div></div>
    <div class="card"><div class="label">NUEVO Inf.</div><div class="value green">{m_n['mean_inf_ms']:.1f}ms</div></div>
  </div>
</div>

<div class="section">
  <h2>02 / Metrics Table</h2>
  <table class="metrics-table">
    <thead>
      <tr><th>Metric</th><th>PROFE (cyan)</th><th>NUEVO (green)</th></tr>
    </thead>
    <tbody>{table_rows_html}</tbody>
  </table>
</div>

<div class="section">
  <h2>03 / Confusion Matrices</h2>
  <div class="cm-wrap">{cm_rows_html}</div>
</div>

<div class="section">
  <h2>04 / Plots</h2>
  <div class="plots-grid">
    <div class="plot-card"><img src="plots/metricas_barras.png"><p>Comparación general de métricas y matrices de confusión</p></div>
    <div class="plot-card"><img src="plots/roc_curve.png"><p>ROC Curve – area bajo la curva</p></div>
    <div class="plot-card"><img src="plots/pr_curve.png"><p>Precision-Recall Curve – Average Precision</p></div>
    <div class="plot-card"><img src="plots/iou_distribution.png"><p>Distribución de IoU sobre detecciones válidas</p></div>
    <div class="plot-card"><img src="plots/inference_time.png"><p>Distribución del tiempo de inferencia por imagen</p></div>
  </div>
</div>

<footer>
  Generado con Ultralytics YOLO · scikit-learn · matplotlib · {len(image_paths)} imágenes evaluadas
</footer>
</body>
</html>"""

with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(html_content)
print(f"\n✅ Reporte HTML: {OUTPUT_HTML}")

# ── Mostrar en notebook ──────────────────────────────────────
display(Video(str(OUTPUT_VIDEO), embed=True))
display(HTML(f'<a href="{OUTPUT_HTML}" target="_blank">📊 Abrir reporte analítico completo</a>'))
print("\n✅ Todo listo. Revisa comparacion_modelos_ball/")

In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
import time
import json
from pathlib import Path
from IPython.display import Video, display, HTML
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import (
    roc_auc_score, confusion_matrix,
    precision_recall_curve, roc_curve, average_precision_score
)
import warnings
warnings.filterwarnings("ignore")

# =========================
# CONFIG
# =========================
MODEL_PROFE  = "best.pt"
MODEL_NUEVO  = "runs_ball/Ball_Padel_LeMar_yolov8s/weights/best.pt"

# ── Dataset (para métricas) ──────────────────────────────────
DATASET     = "data/datasets/Ball_Padel_LeMar_split/val"
IMAGES_DIR  = f"{DATASET}/images"
LABELS_DIR  = f"{DATASET}/labels"

# ── Videos del partido ──────────────────────────────────────
VIDEOS_ROOT = "data/videos"          # subcarpetas permitidas
VIDEO_EXTS  = {".mp4", ".avi", ".mov", ".mkv", ".MP4", ".AVI", ".MOV"}

# ── Salida ───────────────────────────────────────────────────
OUTPUT_DIR  = Path("comparacion_modelos_ball")
VIDEO_DIR   = OUTPUT_DIR / "videos"
PLOT_DIR    = OUTPUT_DIR / "plots"
for d in [VIDEO_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO  = VIDEO_DIR / "SIDE_BY_SIDE_PROFE_VS_NUEVO.mp4"
OUTPUT_HTML   = OUTPUT_DIR / "reporte_analitico.html"

# ── Parámetros ───────────────────────────────────────────────
IMG_SIZE    = 960
CONF        = 0.05
IOU_THRESH  = 0.5
MAX_FRAMES  = None          # None = todos; número = límite total de frames del video
SKIP_FRAMES = 1             # 1 = procesar todos, 2 = saltar 1 de cada 2, etc.
device      = 0

# ─────────────────────────────────────────────────────────────
# COLORES
# ─────────────────────────────────────────────────────────────
COLOR_PROFE = (0, 255, 255)   # cyan  (BGR)
COLOR_NUEVO = (0, 255, 0)     # verde (BGR)
COLOR_GT    = (255, 100, 0)   # naranja (BGR)
HEX_PROFE   = "#00FFFF"
HEX_NUEVO   = "#00FF88"
HEX_BG      = "#0d1117"

# =========================
# MODELOS
# =========================
print("Cargando modelos...")
model_profe = YOLO(MODEL_PROFE)
model_nuevo = YOLO(MODEL_NUEVO)
print("  Clases profe:", model_profe.names)
print("  Clases nuevo:", model_nuevo.names)

# =========================
# HELPERS
# =========================

def get_best_box(result):
    """Devuelve SOLO la detección con mayor confianza."""
    if result.boxes is None or len(result.boxes) == 0:
        return None, 0.0
    confs  = result.boxes.conf.cpu().numpy()
    best_i = int(np.argmax(confs))
    b      = result.boxes[best_i]
    x1, y1, x2, y2 = map(int, b.xyxy[0])
    conf   = float(b.conf[0])
    return [x1, y1, x2, y2], conf


def load_gt_box(label_path, img_w, img_h):
    """Carga la primera bbox del label YOLO (normalizado → píxeles)."""
    if not os.path.exists(str(label_path)):
        return None
    with open(label_path) as f:
        lines = [l.strip() for l in f if l.strip()]
    if not lines:
        return None
    parts = lines[0].split()
    if len(parts) < 5:
        return None
    _, cx, cy, bw, bh = map(float, parts[:5])
    x1 = int((cx - bw / 2) * img_w)
    y1 = int((cy - bh / 2) * img_h)
    x2 = int((cx + bw / 2) * img_w)
    y2 = int((cy + bh / 2) * img_h)
    return [x1, y1, x2, y2]


def iou(boxA, boxB):
    if boxA is None or boxB is None:
        return 0.0
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0


def draw_box(img, box, conf, color, name):
    if box is None:
        return
    x1, y1, x2, y2 = box
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    # punto central
    cx, cy = (x1+x2)//2, (y1+y2)//2
    cv2.circle(img, (cx, cy), 6, color, -1)
    lbl = f"{name} {conf:.2f}"
    (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
    by = max(y1 - 6, th + 4)
    cv2.rectangle(img, (x1, by - th - 4), (x1 + tw + 4, by + 2), color, -1)
    cv2.putText(img, lbl, (x1+2, by), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,0), 2)


def draw_gt(img, box):
    if box is None:
        return
    cv2.rectangle(img, (box[0], box[1]), (box[2], box[3]), COLOR_GT, 2)
    cv2.putText(img, "GT", (box[0], max(box[1]-8, 14)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_GT, 1)


def overlay_header(img, text, color, side="left"):
    """Barra de header semitransparente sobre el frame."""
    overlay = img.copy()
    cv2.rectangle(overlay, (0, 0), (img.shape[1], 38), (0,0,0), -1)
    cv2.addWeighted(overlay, 0.55, img, 0.45, 0, img)
    cv2.putText(img, text, (10, 27), cv2.FONT_HERSHEY_DUPLEX, 0.8, color, 2)


def safe_roc_auc(y_true, y_score):
    """ROC-AUC robusto: falla gracefully si solo hay una clase."""
    y_true = np.array(y_true)
    y_score = np.array(y_score)
    unique = np.unique(y_true)
    if len(unique) < 2:
        # No hay varianza → métrica no aplicable
        return float("nan"), f"(solo clase {unique[0]} presente)"
    try:
        return roc_auc_score(y_true, y_score), ""
    except Exception as e:
        return float("nan"), str(e)


def safe_avg_precision(y_true, y_score):
    y_true = np.array(y_true)
    y_score = np.array(y_score)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    try:
        return average_precision_score(y_true, y_score)
    except Exception:
        return float("nan")

# =========================
# ACUMULADORES DE MÉTRICAS
# =========================
stats = {
    "profe": {"y_true": [], "y_pred": [], "y_score": [],
              "tp": 0, "fp": 0, "fn": 0, "tn": 0,
              "iou_list": [], "conf_list": [], "inf_ms": []},
    "nuevo": {"y_true": [], "y_pred": [], "y_score": [],
              "tp": 0, "fp": 0, "fn": 0, "tn": 0,
              "iou_list": [], "conf_list": [], "inf_ms": []},
}

# =========================
# PASO 1 – MÉTRICAS DESDE
#          EL DATASET DE VAL
# =========================
print("\n📊 Evaluando dataset de validación para métricas...")
valid_ext   = {".jpg", ".jpeg", ".png"}
image_paths = sorted([
    str(p) for p in Path(IMAGES_DIR).rglob("*")
    if p.suffix.lower() in valid_ext
])
print(f"  Total imágenes val: {len(image_paths)}")

t_start = time.time()
for idx, img_path in enumerate(image_paths):
    frame = cv2.imread(img_path)
    if frame is None:
        continue
    h, w = frame.shape[:2]

    stem       = Path(img_path).stem
    label_path = Path(LABELS_DIR) / (stem + ".txt")
    gt_box     = load_gt_box(str(label_path), w, h)
    has_gt     = gt_box is not None

    for key, model in [("profe", model_profe), ("nuevo", model_nuevo)]:
        t0     = time.perf_counter()
        res    = model.predict(frame, imgsz=IMG_SIZE, conf=CONF,
                               device=device, verbose=False)[0]
        inf_ms = (time.perf_counter() - t0) * 1000
        stats[key]["inf_ms"].append(inf_ms)

        box, conf = get_best_box(res)
        detected  = box is not None

        y_score = conf if detected else 0.0
        stats[key]["y_score"].append(y_score)
        stats[key]["y_true"].append(1 if has_gt else 0)
        stats[key]["conf_list"].append(conf if detected else 0.0)

        if has_gt and detected:
            iou_val = iou(gt_box, box)
            stats[key]["iou_list"].append(iou_val)
            if iou_val >= IOU_THRESH:
                stats[key]["tp"] += 1
                stats[key]["y_pred"].append(1)
            else:
                stats[key]["fp"] += 1
                stats[key]["fn"] += 1
                stats[key]["y_pred"].append(0)
        elif has_gt and not detected:
            stats[key]["fn"] += 1
            stats[key]["y_pred"].append(0)
        elif not has_gt and detected:
            stats[key]["fp"] += 1
            stats[key]["y_pred"].append(1)
        else:
            stats[key]["tn"] += 1
            stats[key]["y_pred"].append(0)

    if idx % 100 == 0:
        print(f"  [{idx}/{len(image_paths)}] {time.time()-t_start:.1f}s")

print(f"  ✅ Métricas completadas en {time.time()-t_start:.1f}s")

# =========================
# PASO 2 – VIDEO SIDE-BY-SIDE
#          DESDE VIDEOS REALES
# =========================
print("\n🎬 Buscando videos del partido...")
video_paths = sorted([
    str(p) for p in Path(VIDEOS_ROOT).rglob("*")
    if p.suffix in VIDEO_EXTS
])
print(f"  Videos encontrados: {len(video_paths)}")
for vp in video_paths:
    print(f"    {vp}")

if not video_paths:
    print("  ⚠️  No se encontraron videos en", VIDEOS_ROOT)
else:
    # Detectamos dimensiones del primer video para fijar el writer
    probe = cv2.VideoCapture(video_paths[0])
    src_fps = probe.get(cv2.CAP_PROP_FPS) or 25.0
    src_w   = int(probe.get(cv2.CAP_PROP_FRAME_WIDTH))
    src_h   = int(probe.get(cv2.CAP_PROP_FRAME_HEIGHT))
    probe.release()

    # Output: cada mitad = 640px ancho, alto proporcional (máx 720)
    OUT_W_HALF = 640
    OUT_H      = min(720, int(OUT_W_HALF * src_h / src_w))
    OUT_W      = OUT_W_HALF * 2     # side-by-side
    OUT_FPS    = src_fps

    print(f"  Resolución output: {OUT_W}x{OUT_H} @ {OUT_FPS:.1f} fps")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, OUT_FPS, (OUT_W, OUT_H))

    total_frames_written = 0
    t_video_start = time.time()

    for vid_idx, vid_path in enumerate(video_paths):
        cap = cv2.VideoCapture(vid_path)
        if not cap.isOpened():
            print(f"  ⚠️  No se pudo abrir: {vid_path}")
            continue

        n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        vid_fps  = cap.get(cv2.CAP_PROP_FPS) or 25.0
        vid_name = Path(vid_path).name
        print(f"\n  Procesando [{vid_idx+1}/{len(video_paths)}]: {vid_name} ({n_frames} frames)")

        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if MAX_FRAMES and total_frames_written >= MAX_FRAMES:
                break

            # Skip frames para acelerar si se desea
            if frame_idx % SKIP_FRAMES != 0:
                frame_idx += 1
                continue

            # ── Inferencia ──────────────────────────────────
            res_p = model_profe.predict(frame, imgsz=IMG_SIZE, conf=CONF,
                                        device=device, verbose=False)[0]
            res_n = model_nuevo.predict(frame, imgsz=IMG_SIZE, conf=CONF,
                                        device=device, verbose=False)[0]

            box_p, conf_p = get_best_box(res_p)
            box_n, conf_n = get_best_box(res_n)

            # ── Dibuja en copias independientes ─────────────
            frame_p = frame.copy()
            frame_n = frame.copy()

            draw_box(frame_p, box_p, conf_p, COLOR_PROFE, "PROFE")
            draw_box(frame_n, box_n, conf_n, COLOR_NUEVO, "NUEVO")

            # Header
            label_p = f"PROFE  conf={conf_p:.2f}" if box_p else "PROFE  —"
            label_n = f"NUEVO  conf={conf_n:.2f}" if box_n else "NUEVO  —"
            overlay_header(frame_p, label_p, COLOR_PROFE)
            overlay_header(frame_n, label_n, COLOR_NUEVO)

            # Footer con nombre del video + frame
            info = f"{vid_name}  frame {frame_idx}"
            cv2.putText(frame_p, info, (6, frame_p.shape[0]-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.38, (180,180,180), 1)
            cv2.putText(frame_n, info, (6, frame_n.shape[0]-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.38, (180,180,180), 1)

            # ── Redimensionar y combinar ─────────────────────
            fp = cv2.resize(frame_p, (OUT_W_HALF, OUT_H))
            fn = cv2.resize(frame_n, (OUT_W_HALF, OUT_H))

            # Línea separadora blanca
            combined = np.hstack((fp, fn))
            combined[:, OUT_W_HALF-1:OUT_W_HALF+1] = (220, 220, 220)

            writer.write(combined)
            total_frames_written += 1
            frame_idx += 1

            if frame_idx % 200 == 0:
                elapsed = time.time() - t_video_start
                fps_proc = total_frames_written / max(elapsed, 1)
                print(f"    frame {frame_idx}/{n_frames}  "
                      f"total={total_frames_written}  {fps_proc:.1f} fps proc.")

        cap.release()
        if MAX_FRAMES and total_frames_written >= MAX_FRAMES:
            print("  ⚠️  Límite MAX_FRAMES alcanzado.")
            break

    writer.release()
    elapsed_total = time.time() - t_video_start
    print(f"\n  ✅ Video generado: {OUTPUT_VIDEO}")
    print(f"     Frames totales: {total_frames_written}  |  Tiempo: {elapsed_total:.1f}s")

# ================================================================
# CÁLCULO DE MÉTRICAS
# ================================================================
def compute_metrics(s):
    y_true  = np.array(s["y_true"])
    y_pred  = np.array(s["y_pred"])
    y_score = np.array(s["y_score"])

    tp, fp, fn, tn = s["tp"], s["fp"], s["fn"], s["tn"]

    prec  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec   = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1    = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0.0
    acc   = (tp+tn) / (tp+tn+fp+fn) if (tp+tn+fp+fn) > 0 else 0.0

    # ── ROC-AUC robusto ─────────────────────────────────────
    roc_auc, roc_warn = safe_roc_auc(y_true, y_score)

    # ── Average Precision robusto ───────────────────────────
    avg_prec = safe_avg_precision(y_true, y_score)

    mean_iou  = float(np.mean(s["iou_list"])) if s["iou_list"] else 0.0
    pos_confs = [c for c in s["conf_list"] if c > 0]
    mean_conf = float(np.mean(pos_confs)) if pos_confs else 0.0
    mean_inf  = float(np.mean(s["inf_ms"])) if s["inf_ms"] else 0.0

    return dict(
        accuracy=acc, precision=prec, recall=rec, f1=f1,
        roc_auc=roc_auc, roc_warn=roc_warn,
        avg_precision=avg_prec,
        mean_iou=mean_iou, mean_conf=mean_conf, mean_inf_ms=mean_inf,
        tp=tp, fp=fp, fn=fn, tn=tn,
        y_true=y_true, y_score=y_score, y_pred=y_pred,
        iou_list=s["iou_list"],
    )

m_p = compute_metrics(stats["profe"])
m_n = compute_metrics(stats["nuevo"])

print("\n══════════════════════════════════════════")
print("   MÉTRICAS FINALES")
print("══════════════════════════════════════════")
print(f"{'Métrica':<22} {'PROFE':>10} {'NUEVO':>10}")
print("─" * 46)
for k in ["accuracy","precision","recall","f1","roc_auc","avg_precision","mean_iou","mean_conf","mean_inf_ms"]:
    vp = m_p[k]; vn = m_n[k]
    vp_s = f"{vp:>10.4f}" if not np.isnan(vp) else "       N/A"
    vn_s = f"{vn:>10.4f}" if not np.isnan(vn) else "       N/A"
    warn = ""
    if k == "roc_auc":
        if m_p["roc_warn"]: warn += f" ⚠ PROFE:{m_p['roc_warn']}"
        if m_n["roc_warn"]: warn += f" ⚠ NUEVO:{m_n['roc_warn']}"
    print(f"  {k:<20} {vp_s} {vn_s}{warn}")
for k in ["tp","fp","fn","tn"]:
    print(f"  {k.upper():<20} {m_p[k]:>10} {m_n[k]:>10}")

# ================================================================
# PLOTS
# ================================================================
DARK = "#0d1117"; GREY = "#161b22"; TEXT = "#e6edf3"
ACCENT = "#58a6ff"
plt.rcParams.update({
    "figure.facecolor": DARK, "axes.facecolor": GREY,
    "axes.edgecolor": "#30363d", "axes.labelcolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT,
    "text.color": TEXT, "grid.color": "#21262d",
    "font.family": "monospace",
})

# ── 1. Barras de métricas ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(DARK)
metrics_keys = ["accuracy","precision","recall","f1","roc_auc","avg_precision","mean_iou"]
labels_k     = ["Accuracy","Precision","Recall","F1","ROC-AUC","mAP","Mean IoU"]
vp = [m_p[k] if not np.isnan(m_p[k]) else 0 for k in metrics_keys]
vn = [m_n[k] if not np.isnan(m_n[k]) else 0 for k in metrics_keys]

x  = np.arange(len(labels_k))
w  = 0.35
ax = axes[0]
ax.bar(x - w/2, vp, w, label="PROFE", color=HEX_PROFE, alpha=0.9)
ax.bar(x + w/2, vn, w, label="NUEVO", color=HEX_NUEVO, alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(labels_k, rotation=30, ha="right", fontsize=8)
ax.set_ylim(0, 1.15); ax.set_title("Comparación de Métricas", fontsize=11, color=TEXT)
ax.legend(facecolor=GREY, edgecolor="#30363d"); ax.grid(axis="y", alpha=0.4)

# Etiqueta N/A si roc_auc es nan
for i, k in enumerate(metrics_keys):
    if k == "roc_auc":
        if np.isnan(m_p["roc_auc"]):
            ax.text(i - w/2, 0.05, "N/A", ha="center", fontsize=7, color=HEX_PROFE)
        if np.isnan(m_n["roc_auc"]):
            ax.text(i + w/2, 0.05, "N/A", ha="center", fontsize=7, color=HEX_NUEVO)

# Confusion matrices en axes[1]
ax2 = axes[1]; ax2.axis("off")
table_data = [["","Pred 0","Pred 1"],
              ["Act 0",f"TN={m_p['tn']}",f"FP={m_p['fp']}"],
              ["Act 1",f"FN={m_p['fn']}",f"TP={m_p['tp']}"]]
table2     = [["","Pred 0","Pred 1"],
              ["Act 0",f"TN={m_n['tn']}",f"FP={m_n['fp']}"],
              ["Act 1",f"FN={m_n['fn']}",f"TP={m_n['tp']}"]]
y_base = 0.85
for title, tdata, color in [
        ("PROFE – Confusion Matrix", table_data, HEX_PROFE),
        ("NUEVO – Confusion Matrix", table2,     HEX_NUEVO)]:
    ax2.text(0.02 if "PROFE" in title else 0.52, y_base+0.08,
             title, transform=ax2.transAxes, color=color, fontsize=9, fontweight="bold")
    for r, row in enumerate(tdata):
        for c, val in enumerate(row):
            xp = (0.02 if "PROFE" in title else 0.52) + c * 0.14
            yp = y_base - r * 0.22
            bg = "#1f2937" if r==0 or c==0 else (
                 "#134e2a" if (r==2 and c==2) else
                 "#7f1d1d" if (r==1 and c==2 or r==2 and c==1) else "#1f2937")
            ax2.text(xp, yp, val, transform=ax2.transAxes,
                     ha="center", va="center", fontsize=8, color=TEXT,
                     bbox=dict(facecolor=bg, edgecolor="#30363d", boxstyle="round,pad=0.3"))

plt.tight_layout()
bar_path = PLOT_DIR / "metricas_barras.png"
plt.savefig(bar_path, dpi=130, bbox_inches="tight", facecolor=DARK); plt.close()
print(f"  Saved: {bar_path}")

# ── 2. ROC Curve ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(DARK); ax.set_facecolor(GREY)
for (label, m, color) in [("PROFE", m_p, HEX_PROFE), ("NUEVO", m_n, HEX_NUEVO)]:
    if np.isnan(m["roc_auc"]):
        ax.text(0.5, 0.5 if label=="PROFE" else 0.4,
                f"{label}: ROC-AUC N/A\n(una sola clase en y_true)",
                ha="center", color=color, fontsize=9, transform=ax.transAxes)
        continue
    try:
        fpr, tpr, _ = roc_curve(m["y_true"], m["y_score"])
        ax.plot(fpr, tpr, color=color, lw=2,
                label=f"{label}  AUC={m['roc_auc']:.3f}")
        ax.fill_between(fpr, tpr, alpha=0.08, color=color)
    except Exception as e:
        print(f"  ROC plot error ({label}): {e}")
ax.plot([0,1],[0,1],"--",color="#30363d")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve", color=TEXT)
handles = ax.get_legend_handles_labels()
if handles[0]: ax.legend(facecolor=GREY, edgecolor="#30363d")
ax.grid(alpha=0.3)
roc_path = PLOT_DIR / "roc_curve.png"
plt.savefig(roc_path, dpi=130, bbox_inches="tight", facecolor=DARK); plt.close()
print(f"  Saved: {roc_path}")

# ── 3. Precision-Recall Curve ────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(DARK); ax.set_facecolor(GREY)
for (label, m, color) in [("PROFE", m_p, HEX_PROFE), ("NUEVO", m_n, HEX_NUEVO)]:
    if np.isnan(m["avg_precision"]):
        continue
    try:
        p, r, _ = precision_recall_curve(m["y_true"], m["y_score"])
        ax.plot(r, p, color=color, lw=2,
                label=f"{label}  AP={m['avg_precision']:.3f}")
        ax.fill_between(r, p, alpha=0.08, color=color)
    except Exception as e:
        print(f"  PR plot error ({label}): {e}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve", color=TEXT)
handles = ax.get_legend_handles_labels()
if handles[0]: ax.legend(facecolor=GREY, edgecolor="#30363d")
ax.grid(alpha=0.3)
pr_path = PLOT_DIR / "pr_curve.png"
plt.savefig(pr_path, dpi=130, bbox_inches="tight", facecolor=DARK); plt.close()
print(f"  Saved: {pr_path}")

# ── 4. IoU Distribution ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(DARK); ax.set_facecolor(GREY)
if m_p["iou_list"]:
    ax.hist(m_p["iou_list"], bins=30, color=HEX_PROFE, alpha=0.7,
            label=f"PROFE  μ={m_p['mean_iou']:.3f}")
if m_n["iou_list"]:
    ax.hist(m_n["iou_list"], bins=30, color=HEX_NUEVO, alpha=0.7,
            label=f"NUEVO  μ={m_n['mean_iou']:.3f}")
ax.axvline(IOU_THRESH, color="white", linestyle="--", alpha=0.6, label=f"IoU≥{IOU_THRESH}")
ax.set_xlabel("IoU"); ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de IoU (detecciones sobre GT)", color=TEXT)
ax.legend(facecolor=GREY, edgecolor="#30363d"); ax.grid(alpha=0.3)
iou_path = PLOT_DIR / "iou_distribution.png"
plt.savefig(iou_path, dpi=130, bbox_inches="tight", facecolor=DARK); plt.close()
print(f"  Saved: {iou_path}")

# ── 5. Inference time ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(DARK); ax.set_facecolor(GREY)
ax.hist(stats["profe"]["inf_ms"], bins=40, color=HEX_PROFE, alpha=0.7,
        label=f"PROFE  μ={m_p['mean_inf_ms']:.1f}ms")
ax.hist(stats["nuevo"]["inf_ms"], bins=40, color=HEX_NUEVO, alpha=0.7,
        label=f"NUEVO  μ={m_n['mean_inf_ms']:.1f}ms")
ax.set_xlabel("Tiempo inferencia (ms)"); ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de Tiempo de Inferencia", color=TEXT)
ax.legend(facecolor=GREY, edgecolor="#30363d"); ax.grid(alpha=0.3)
inf_path = PLOT_DIR / "inference_time.png"
plt.savefig(inf_path, dpi=130, bbox_inches="tight", facecolor=DARK); plt.close()
print(f"  Saved: {inf_path}")

# ================================================================
# REPORTE HTML
# ================================================================
def fmt_val(name, v):
    if np.isnan(v): return "N/A"
    if "ms" in name:    return f"{v:.1f} ms"
    if "conf" in name.lower(): return f"{v:.3f}"
    return f"{v*100:.2f}%"

winner = lambda vp, vn: ("🏆","—") if vp>vn else ("—","🏆") if vn>vp else ("—","—")

rows = [
    ("Accuracy",            m_p["accuracy"],       m_n["accuracy"],       True),
    ("Precision",           m_p["precision"],      m_n["precision"],      True),
    ("Recall",              m_p["recall"],         m_n["recall"],         True),
    ("F1-Score",            m_p["f1"],             m_n["f1"],             True),
    ("ROC-AUC",             m_p["roc_auc"],        m_n["roc_auc"],        True),
    ("Avg Precision (mAP)", m_p["avg_precision"],  m_n["avg_precision"],  True),
    ("Mean IoU",            m_p["mean_iou"],       m_n["mean_iou"],       True),
    ("Mean Confidence",     m_p["mean_conf"],      m_n["mean_conf"],      True),
    ("Inf. Time (ms)",      m_p["mean_inf_ms"],    m_n["mean_inf_ms"],    False),
]

table_rows_html = ""
for name, vp, vn, higher_better in rows:
    wp, wn = ("—","—")
    if not (np.isnan(vp) or np.isnan(vn)):
        wp, wn = winner(vp, vn) if higher_better else winner(-vp, -vn)
    vp_s = fmt_val(name, vp)
    vn_s = fmt_val(name, vn)
    table_rows_html += f"""
        <tr>
          <td>{name}</td>
          <td class="val cyan">{vp_s} {wp}</td>
          <td class="val green">{vn_s} {wn}</td>
        </tr>"""

cm_rows_html = ""
for label, m, color in [("PROFE", m_p, "cyan"), ("NUEVO", m_n, "green")]:
    cm_rows_html += f"""
    <div class="cm-block">
      <h3 class="{color}">{label} – Confusion Matrix</h3>
      <table class="cm-table">
        <tr><th></th><th>Pred 0</th><th>Pred 1</th></tr>
        <tr><td>Act 0</td><td class="tn">TN<br>{m['tn']}</td><td class="fp">FP<br>{m['fp']}</td></tr>
        <tr><td>Act 1</td><td class="fn">FN<br>{m['fn']}</td><td class="tp">TP<br>{m['tp']}</td></tr>
      </table>
    </div>"""

roc_note_p = f"<br><small>⚠ {m_p['roc_warn']}</small>" if m_p.get("roc_warn") else ""
roc_note_n = f"<br><small>⚠ {m_n['roc_warn']}</small>" if m_n.get("roc_warn") else ""

html_content = f"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<title>Reporte Analítico – Comparación de Modelos</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600;700&family=Space+Grotesk:wght@400;600;700&display=swap');
  *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
  :root {{
    --bg:#0d1117;--surface:#161b22;--border:#30363d;
    --text:#e6edf3;--muted:#8b949e;
    --cyan:#00ffff;--green:#00ff88;--blue:#58a6ff;
    --red:#ff6b6b;--yellow:#ffd700;
  }}
  body {{ background:var(--bg);color:var(--text);font-family:'Space Grotesk',sans-serif;padding:2rem; }}
  h1 {{ font-family:'JetBrains Mono',monospace;font-size:1.6rem;color:var(--cyan);margin-bottom:.3rem; }}
  .subtitle {{ color:var(--muted);font-size:.85rem;margin-bottom:2rem; }}
  .section {{ margin-bottom:2.5rem; }}
  h2 {{ font-family:'JetBrains Mono',monospace;font-size:1rem;color:var(--blue);border-bottom:1px solid var(--border);padding-bottom:.5rem;margin-bottom:1rem; }}
  h3 {{ font-size:.9rem;margin-bottom:.5rem; }}
  .cyan {{ color:var(--cyan); }} .green {{ color:var(--green); }}
  .metrics-table {{ width:100%;border-collapse:collapse;font-family:'JetBrains Mono',monospace;font-size:.82rem; }}
  .metrics-table th {{ background:var(--surface);color:var(--muted);padding:.5rem 1rem;text-align:left;border:1px solid var(--border); }}
  .metrics-table td {{ padding:.5rem 1rem;border:1px solid var(--border); }}
  .metrics-table tr:nth-child(even) td {{ background:#0f1419; }}
  .val {{ text-align:right;font-weight:600;font-size:.9rem; }}
  .metrics-table td.cyan {{ color:var(--cyan); }} .metrics-table td.green {{ color:var(--green); }}
  .cm-wrap {{ display:flex;gap:2rem;flex-wrap:wrap; }}
  .cm-block {{ background:var(--surface);border:1px solid var(--border);border-radius:8px;padding:1rem 1.5rem; }}
  .cm-table {{ border-collapse:collapse;font-family:'JetBrains Mono',monospace;font-size:.85rem;margin-top:.5rem; }}
  .cm-table th,.cm-table td {{ border:1px solid var(--border);padding:.5rem 1rem;text-align:center; }}
  .cm-table th {{ background:#21262d;color:var(--muted); }}
  .tp {{ background:#0d2818;color:#4ade80;font-weight:700; }}
  .tn {{ background:#1a2540;color:var(--blue);font-weight:700; }}
  .fp {{ background:#3b1d1d;color:var(--red);font-weight:700; }}
  .fn {{ background:#3b2600;color:var(--yellow);font-weight:700; }}
  .plots-grid {{ display:grid;grid-template-columns:repeat(auto-fit,minmax(420px,1fr));gap:1rem; }}
  .plot-card {{ background:var(--surface);border:1px solid var(--border);border-radius:8px;overflow:hidden; }}
  .plot-card img {{ width:100%;display:block; }}
  .plot-card p {{ padding:.4rem .8rem;font-size:.75rem;color:var(--muted); }}
  .cards {{ display:flex;gap:1rem;flex-wrap:wrap;margin-bottom:1.5rem; }}
  .card {{ background:var(--surface);border:1px solid var(--border);border-radius:8px;padding:1rem 1.5rem;min-width:160px; }}
  .card .label {{ font-size:.7rem;color:var(--muted);text-transform:uppercase;letter-spacing:.08em; }}
  .card .value {{ font-size:1.4rem;font-weight:700;font-family:'JetBrains Mono',monospace;margin-top:.2rem; }}
  .note {{ font-size:.75rem;color:var(--muted);font-style:italic; }}
  footer {{ margin-top:3rem;font-size:.75rem;color:var(--muted);border-top:1px solid var(--border);padding-top:1rem; }}
</style>
</head>
<body>
<h1>🎾 Model Analytics Report</h1>
<p class="subtitle">PROFE vs NUEVO · Ball Padel Detection · IoU threshold = {IOU_THRESH} · {len(image_paths)} imágenes evaluadas</p>

<div class="section">
  <h2>01 / Summary Scorecards</h2>
  <div class="cards">
    <div class="card"><div class="label">PROFE F1</div><div class="value cyan">{m_p['f1']*100:.1f}%</div></div>
    <div class="card"><div class="label">NUEVO F1</div><div class="value green">{m_n['f1']*100:.1f}%</div></div>
    <div class="card"><div class="label">PROFE ROC-AUC</div><div class="value cyan">{"N/A" if np.isnan(m_p['roc_auc']) else f"{m_p['roc_auc']:.3f}"}{roc_note_p}</div></div>
    <div class="card"><div class="label">NUEVO ROC-AUC</div><div class="value green">{"N/A" if np.isnan(m_n['roc_auc']) else f"{m_n['roc_auc']:.3f}"}{roc_note_n}</div></div>
    <div class="card"><div class="label">PROFE Mean IoU</div><div class="value cyan">{m_p['mean_iou']:.3f}</div></div>
    <div class="card"><div class="label">NUEVO Mean IoU</div><div class="value green">{m_n['mean_iou']:.3f}</div></div>
    <div class="card"><div class="label">PROFE Inf.</div><div class="value cyan">{m_p['mean_inf_ms']:.1f}ms</div></div>
    <div class="card"><div class="label">NUEVO Inf.</div><div class="value green">{m_n['mean_inf_ms']:.1f}ms</div></div>
  </div>
  <p class="note">⚠ ROC-AUC = N/A cuando el dataset de validación tiene solo una clase (e.g., todas las imágenes tienen pelota). Es un problema del split, no del modelo.</p>
</div>

<div class="section">
  <h2>02 / Metrics Table</h2>
  <table class="metrics-table">
    <thead><tr><th>Metric</th><th>PROFE (cyan)</th><th>NUEVO (green)</th></tr></thead>
    <tbody>{table_rows_html}</tbody>
  </table>
</div>

<div class="section">
  <h2>03 / Confusion Matrices</h2>
  <div class="cm-wrap">{cm_rows_html}</div>
</div>

<div class="section">
  <h2>04 / Plots</h2>
  <div class="plots-grid">
    <div class="plot-card"><img src="plots/metricas_barras.png"><p>Comparación general de métricas y matrices de confusión</p></div>
    <div class="plot-card"><img src="plots/roc_curve.png"><p>ROC Curve – area bajo la curva</p></div>
    <div class="plot-card"><img src="plots/pr_curve.png"><p>Precision-Recall Curve – Average Precision</p></div>
    <div class="plot-card"><img src="plots/iou_distribution.png"><p>Distribución de IoU sobre detecciones válidas</p></div>
    <div class="plot-card"><img src="plots/inference_time.png"><p>Distribución del tiempo de inferencia por imagen</p></div>
  </div>
</div>

<footer>
  Generado con Ultralytics YOLO · scikit-learn · matplotlib · Videos: {len(video_paths)} archivos en {VIDEOS_ROOT}
</footer>
</body>
</html>"""

with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(html_content)
print(f"\n✅ Reporte HTML: {OUTPUT_HTML}")

# ── Mostrar en notebook ──────────────────────────────────────
try:
    display(Video(str(OUTPUT_VIDEO), embed=True))
    display(HTML(f'<a href="{OUTPUT_HTML}" target="_blank">📊 Abrir reporte analítico completo</a>'))
except Exception:
    pass
print("\n✅ Todo listo. Revisa comparacion_modelos_ball/")

In [ ]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
import time
import json
from pathlib import Path
from IPython.display import Video, display, HTML
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import (
    roc_auc_score, confusion_matrix,
    precision_recall_curve, roc_curve, average_precision_score
)
import warnings
warnings.filterwarnings("ignore")

# =========================
# CONFIG
# =========================
MODEL_PROFE  = "best.pt"
MODEL_NUEVO  = "runs_ball/Ball_Padel_LeMar_yolov8s/weights/best.pt"

# ── Dataset (para métricas) ──────────────────────────────────
DATASET     = "data/datasets/Ball_Padel_LeMar_split/val"
IMAGES_DIR  = f"{DATASET}/images"
LABELS_DIR  = f"{DATASET}/labels"

# ── Videos del partido ──────────────────────────────────────
VIDEOS_ROOT = "data/videos"          # subcarpetas permitidas
VIDEO_EXTS  = {".mp4", ".avi", ".mov", ".mkv", ".MP4", ".AVI", ".MOV"}

# ── Salida ───────────────────────────────────────────────────
OUTPUT_DIR  = Path("comparacion_modelos_ball")
VIDEO_DIR   = OUTPUT_DIR / "videos"
PLOT_DIR    = OUTPUT_DIR / "plots"
for d in [VIDEO_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OUTPUT_VIDEO  = VIDEO_DIR / "SIDE_BY_SIDE_PROFE_VS_NUEVO.mp4"
OUTPUT_HTML   = OUTPUT_DIR / "reporte_analitico.html"

# ── Parámetros ───────────────────────────────────────────────
IMG_SIZE    = 960
CONF        = 0.05
IOU_THRESH  = 0.5
MAX_FRAMES  = None          # None = todos; número = límite total de frames del video
SKIP_FRAMES = 1             # 1 = procesar todos, 2 = saltar 1 de cada 2, etc.
device      = 0

# ─────────────────────────────────────────────────────────────
# COLORES
# ─────────────────────────────────────────────────────────────
COLOR_PROFE = (0, 255, 255)   # cyan  (BGR)
COLOR_NUEVO = (0, 255, 0)     # verde (BGR)
COLOR_GT    = (255, 100, 0)   # naranja (BGR)
HEX_PROFE   = "#00FFFF"
HEX_NUEVO   = "#00FF88"
HEX_BG      = "#0d1117"

# =========================
# MODELOS
# =========================
print("Cargando modelos...")
model_profe = YOLO(MODEL_PROFE)
model_nuevo = YOLO(MODEL_NUEVO)
print("  Clases profe:", model_profe.names)
print("  Clases nuevo:", model_nuevo.names)

# =========================
# HELPERS
# =========================

def get_best_box(result):
    """Devuelve SOLO la detección con mayor confianza."""
    if result.boxes is None or len(result.boxes) == 0:
        return None, 0.0
    confs  = result.boxes.conf.cpu().numpy()
    best_i = int(np.argmax(confs))
    b      = result.boxes[best_i]
    x1, y1, x2, y2 = map(int, b.xyxy[0])
    conf   = float(b.conf[0])
    return [x1, y1, x2, y2], conf


def load_gt_box(label_path, img_w, img_h):
    """Carga la primera bbox del label YOLO (normalizado → píxeles)."""
    if not os.path.exists(str(label_path)):
        return None
    with open(label_path) as f:
        lines = [l.strip() for l in f if l.strip()]
    if not lines:
        return None
    parts = lines[0].split()
    if len(parts) < 5:
        return None
    _, cx, cy, bw, bh = map(float, parts[:5])
    x1 = int((cx - bw / 2) * img_w)
    y1 = int((cy - bh / 2) * img_h)
    x2 = int((cx + bw / 2) * img_w)
    y2 = int((cy + bh / 2) * img_h)
    return [x1, y1, x2, y2]


def iou(boxA, boxB):
    if boxA is None or boxB is None:
        return 0.0
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0


def draw_box(img, box, conf, color, name):
    if box is None:
        return
    x1, y1, x2, y2 = box
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    # punto central
    cx, cy = (x1+x2)//2, (y1+y2)//2
    cv2.circle(img, (cx, cy), 6, color, -1)
    lbl = f"{name} {conf:.2f}"
    (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
    by = max(y1 - 6, th + 4)
    cv2.rectangle(img, (x1, by - th - 4), (x1 + tw + 4, by + 2), color, -1)
    cv2.putText(img, lbl, (x1+2, by), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,0), 2)


def draw_gt(img, box):
    if box is None:
        return
    cv2.rectangle(img, (box[0], box[1]), (box[2], box[3]), COLOR_GT, 2)
    cv2.putText(img, "GT", (box[0], max(box[1]-8, 14)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_GT, 1)


def overlay_header(img, text, color, side="left"):
    """Barra de header semitransparente sobre el frame."""
    overlay = img.copy()
    cv2.rectangle(overlay, (0, 0), (img.shape[1], 38), (0,0,0), -1)
    cv2.addWeighted(overlay, 0.55, img, 0.45, 0, img)
    cv2.putText(img, text, (10, 27), cv2.FONT_HERSHEY_DUPLEX, 0.8, color, 2)


def safe_roc_auc(y_true, y_score):
    """ROC-AUC robusto: falla gracefully si solo hay una clase."""
    y_true = np.array(y_true)
    y_score = np.array(y_score)
    unique = np.unique(y_true)
    if len(unique) < 2:
        # No hay varianza → métrica no aplicable
        return float("nan"), f"(solo clase {unique[0]} presente)"
    try:
        return roc_auc_score(y_true, y_score), ""
    except Exception as e:
        return float("nan"), str(e)


def safe_avg_precision(y_true, y_score):
    y_true = np.array(y_true)
    y_score = np.array(y_score)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    try:
        return average_precision_score(y_true, y_score)
    except Exception:
        return float("nan")

# =========================
# ACUMULADORES DE MÉTRICAS
# =========================
stats = {
    "profe": {"y_true": [], "y_pred": [], "y_score": [],
              "tp": 0, "fp": 0, "fn": 0, "tn": 0,
              "iou_list": [], "conf_list": [], "inf_ms": []},
    "nuevo": {"y_true": [], "y_pred": [], "y_score": [],
              "tp": 0, "fp": 0, "fn": 0, "tn": 0,
              "iou_list": [], "conf_list": [], "inf_ms": []},
}

# =========================
# PASO 1 – NEGATIVOS SINTÉTICOS
#          desde los videos
# =========================
# Cuántos negativos generar (aprox. mismo nº que positivos del val set,
# o el valor fijo que prefieras)
NEG_PER_VIDEO   = 30        # frames negativos a intentar extraer por video
NEG_CONF_MAX    = 0.15      # si ambos modelos detectan con conf > esto → no es negativo confiable
NEGS_DIR        = OUTPUT_DIR / "negatives"
NEGS_DIR.mkdir(parents=True, exist_ok=True)

print("\n🔴 Generando negativos sintéticos desde videos...")
video_paths_neg = sorted([
    str(p) for p in Path(VIDEOS_ROOT).rglob("*")
    if p.suffix in VIDEO_EXTS
])

neg_image_paths = []

for vid_path in video_paths_neg:
    cap = cv2.VideoCapture(vid_path)
    if not cap.isOpened():
        continue
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    vid_name = Path(vid_path).stem
    found_negs = 0

    # Muestreamos frames distribuidos uniformemente por el video
    sample_indices = np.linspace(0, n_frames - 1, min(n_frames, NEG_PER_VIDEO * 10), dtype=int)

    for fi in sample_indices:
        if found_negs >= NEG_PER_VIDEO:
            break
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if not ret or frame is None:
            continue

        # Inferencia rápida en ambos modelos
        res_p = model_profe.predict(frame, imgsz=IMG_SIZE, conf=NEG_CONF_MAX,
                                    device=device, verbose=False)[0]
        res_n = model_nuevo.predict(frame, imgsz=IMG_SIZE, conf=NEG_CONF_MAX,
                                    device=device, verbose=False)[0]

        det_p = res_p.boxes is not None and len(res_p.boxes) > 0
        det_n = res_n.boxes is not None and len(res_n.boxes) > 0

        # Negativo confiable = ninguno de los dos detecta nada
        if not det_p and not det_n:
            neg_path = NEGS_DIR / f"{vid_name}_f{fi:06d}.jpg"
            cv2.imwrite(str(neg_path), frame)
            neg_image_paths.append(str(neg_path))
            found_negs += 1

    cap.release()
    print(f"  {Path(vid_path).name}: {found_negs} negativos extraídos")

print(f"  ✅ Total negativos sintéticos: {len(neg_image_paths)}")

# =========================
# PASO 2 – MÉTRICAS DESDE
#          EL DATASET DE VAL
#          + negativos
# =========================
print("\n📊 Evaluando dataset de validación para métricas...")
valid_ext   = {".jpg", ".jpeg", ".png"}
pos_image_paths = sorted([
    str(p) for p in Path(IMAGES_DIR).rglob("*")
    if p.suffix.lower() in valid_ext
])

# Mezclamos positivos (con GT) y negativos sintéticos (sin GT)
# Los negativos no tienen label file → load_gt_box devuelve None → has_gt=False
image_paths = pos_image_paths + neg_image_paths
print(f"  Positivos (val): {len(pos_image_paths)}")
print(f"  Negativos (sint): {len(neg_image_paths)}")
print(f"  Total: {len(image_paths)}")

print(f"\n  Evaluando {len(image_paths)} imágenes (positivos + negativos)...")
t_start = time.time()
for idx, img_path in enumerate(image_paths):
    frame = cv2.imread(img_path)
    if frame is None:
        continue
    h, w = frame.shape[:2]

    stem       = Path(img_path).stem
    label_path = Path(LABELS_DIR) / (stem + ".txt")
    gt_box     = load_gt_box(str(label_path), w, h)
    has_gt     = gt_box is not None

    for key, model in [("profe", model_profe), ("nuevo", model_nuevo)]:
        t0     = time.perf_counter()
        res    = model.predict(frame, imgsz=IMG_SIZE, conf=CONF,
                               device=device, verbose=False)[0]
        inf_ms = (time.perf_counter() - t0) * 1000
        stats[key]["inf_ms"].append(inf_ms)

        box, conf = get_best_box(res)
        detected  = box is not None

        y_score = conf if detected else 0.0
        stats[key]["y_score"].append(y_score)
        stats[key]["y_true"].append(1 if has_gt else 0)
        stats[key]["conf_list"].append(conf if detected else 0.0)

        if has_gt and detected:
            iou_val = iou(gt_box, box)
            stats[key]["iou_list"].append(iou_val)
            if iou_val >= IOU_THRESH:
                stats[key]["tp"] += 1
                stats[key]["y_pred"].append(1)
            else:
                stats[key]["fp"] += 1
                stats[key]["fn"] += 1
                stats[key]["y_pred"].append(0)
        elif has_gt and not detected:
            stats[key]["fn"] += 1
            stats[key]["y_pred"].append(0)
        elif not has_gt and detected:
            stats[key]["fp"] += 1
            stats[key]["y_pred"].append(1)
        else:
            stats[key]["tn"] += 1
            stats[key]["y_pred"].append(0)

    if idx % 100 == 0:
        n_pos = sum(stats["profe"]["y_true"]); n_neg = len(stats["profe"]["y_true"]) - n_pos
        print(f"  [{idx}/{len(image_paths)}] pos={n_pos} neg={n_neg}  {time.time()-t_start:.1f}s")

print(f"  ✅ Métricas completadas en {time.time()-t_start:.1f}s")

# =========================
# PASO 2 – VIDEO SIDE-BY-SIDE
#          DESDE VIDEOS REALES
# =========================
print("\n🎬 Buscando videos del partido...")
video_paths = sorted([
    str(p) for p in Path(VIDEOS_ROOT).rglob("*")
    if p.suffix in VIDEO_EXTS
])
print(f"  Videos encontrados: {len(video_paths)}")
for vp in video_paths:
    print(f"    {vp}")

if not video_paths:
    print("  ⚠️  No se encontraron videos en", VIDEOS_ROOT)
else:
    # Detectamos dimensiones del primer video para fijar el writer
    probe = cv2.VideoCapture(video_paths[0])
    src_fps = probe.get(cv2.CAP_PROP_FPS) or 25.0
    src_w   = int(probe.get(cv2.CAP_PROP_FRAME_WIDTH))
    src_h   = int(probe.get(cv2.CAP_PROP_FRAME_HEIGHT))
    probe.release()

    # Output: cada mitad = 640px ancho, alto proporcional (máx 720)
    OUT_W_HALF = 640
    OUT_H      = min(720, int(OUT_W_HALF * src_h / src_w))
    OUT_W      = OUT_W_HALF * 2     # side-by-side
    OUT_FPS    = src_fps

    print(f"  Resolución output: {OUT_W}x{OUT_H} @ {OUT_FPS:.1f} fps")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, OUT_FPS, (OUT_W, OUT_H))

    total_frames_written = 0
    t_video_start = time.time()

    for vid_idx, vid_path in enumerate(video_paths):
        cap = cv2.VideoCapture(vid_path)
        if not cap.isOpened():
            print(f"  ⚠️  No se pudo abrir: {vid_path}")
            continue

        n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        vid_fps  = cap.get(cv2.CAP_PROP_FPS) or 25.0
        vid_name = Path(vid_path).name
        print(f"\n  Procesando [{vid_idx+1}/{len(video_paths)}]: {vid_name} ({n_frames} frames)")

        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if MAX_FRAMES and total_frames_written >= MAX_FRAMES:
                break

            # Skip frames para acelerar si se desea
            if frame_idx % SKIP_FRAMES != 0:
                frame_idx += 1
                continue

            # ── Inferencia ──────────────────────────────────
            res_p = model_profe.predict(frame, imgsz=IMG_SIZE, conf=CONF,
                                        device=device, verbose=False)[0]
            res_n = model_nuevo.predict(frame, imgsz=IMG_SIZE, conf=CONF,
                                        device=device, verbose=False)[0]

            box_p, conf_p = get_best_box(res_p)
            box_n, conf_n = get_best_box(res_n)

            # ── Dibuja en copias independientes ─────────────
            frame_p = frame.copy()
            frame_n = frame.copy()

            draw_box(frame_p, box_p, conf_p, COLOR_PROFE, "PROFE")
            draw_box(frame_n, box_n, conf_n, COLOR_NUEVO, "NUEVO")

            # Header
            label_p = f"PROFE  conf={conf_p:.2f}" if box_p else "PROFE  —"
            label_n = f"NUEVO  conf={conf_n:.2f}" if box_n else "NUEVO  —"
            overlay_header(frame_p, label_p, COLOR_PROFE)
            overlay_header(frame_n, label_n, COLOR_NUEVO)

            # Footer con nombre del video + frame
            info = f"{vid_name}  frame {frame_idx}"
            cv2.putText(frame_p, info, (6, frame_p.shape[0]-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.38, (180,180,180), 1)
            cv2.putText(frame_n, info, (6, frame_n.shape[0]-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.38, (180,180,180), 1)

            # ── Redimensionar y combinar ─────────────────────
            fp = cv2.resize(frame_p, (OUT_W_HALF, OUT_H))
            fn = cv2.resize(frame_n, (OUT_W_HALF, OUT_H))

            # Línea separadora blanca
            combined = np.hstack((fp, fn))
            combined[:, OUT_W_HALF-1:OUT_W_HALF+1] = (220, 220, 220)

            writer.write(combined)
            total_frames_written += 1
            frame_idx += 1

            if frame_idx % 200 == 0:
                elapsed = time.time() - t_video_start
                fps_proc = total_frames_written / max(elapsed, 1)
                print(f"    frame {frame_idx}/{n_frames}  "
                      f"total={total_frames_written}  {fps_proc:.1f} fps proc.")

        cap.release()
        if MAX_FRAMES and total_frames_written >= MAX_FRAMES:
            print("  ⚠️  Límite MAX_FRAMES alcanzado.")
            break

    writer.release()
    elapsed_total = time.time() - t_video_start
    print(f"\n  ✅ Video generado: {OUTPUT_VIDEO}")
    print(f"     Frames totales: {total_frames_written}  |  Tiempo: {elapsed_total:.1f}s")

# ================================================================
# CÁLCULO DE MÉTRICAS
# ================================================================
def compute_metrics(s):
    y_true  = np.array(s["y_true"])
    y_pred  = np.array(s["y_pred"])
    y_score = np.array(s["y_score"])

    tp, fp, fn, tn = s["tp"], s["fp"], s["fn"], s["tn"]

    prec  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec   = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1    = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0.0
    acc   = (tp+tn) / (tp+tn+fp+fn) if (tp+tn+fp+fn) > 0 else 0.0

    # ── ROC-AUC robusto ─────────────────────────────────────
    roc_auc, roc_warn = safe_roc_auc(y_true, y_score)

    # ── Average Precision robusto ───────────────────────────
    avg_prec = safe_avg_precision(y_true, y_score)

    mean_iou  = float(np.mean(s["iou_list"])) if s["iou_list"] else 0.0
    pos_confs = [c for c in s["conf_list"] if c > 0]
    mean_conf = float(np.mean(pos_confs)) if pos_confs else 0.0
    mean_inf  = float(np.mean(s["inf_ms"])) if s["inf_ms"] else 0.0

    return dict(
        accuracy=acc, precision=prec, recall=rec, f1=f1,
        roc_auc=roc_auc, roc_warn=roc_warn,
        avg_precision=avg_prec,
        mean_iou=mean_iou, mean_conf=mean_conf, mean_inf_ms=mean_inf,
        tp=tp, fp=fp, fn=fn, tn=tn,
        y_true=y_true, y_score=y_score, y_pred=y_pred,
        iou_list=s["iou_list"],
    )

m_p = compute_metrics(stats["profe"])
m_n = compute_metrics(stats["nuevo"])

print("\n══════════════════════════════════════════")
print("   MÉTRICAS FINALES")
print("══════════════════════════════════════════")
print(f"{'Métrica':<22} {'PROFE':>10} {'NUEVO':>10}")
print("─" * 46)
for k in ["accuracy","precision","recall","f1","roc_auc","avg_precision","mean_iou","mean_conf","mean_inf_ms"]:
    vp = m_p[k]; vn = m_n[k]
    vp_s = f"{vp:>10.4f}" if not np.isnan(vp) else "       N/A"
    vn_s = f"{vn:>10.4f}" if not np.isnan(vn) else "       N/A"
    warn = ""
    if k == "roc_auc":
        if m_p["roc_warn"]: warn += f" ⚠ PROFE:{m_p['roc_warn']}"
        if m_n["roc_warn"]: warn += f" ⚠ NUEVO:{m_n['roc_warn']}"
    print(f"  {k:<20} {vp_s} {vn_s}{warn}")
for k in ["tp","fp","fn","tn"]:
    print(f"  {k.upper():<20} {m_p[k]:>10} {m_n[k]:>10}")

# ================================================================
# PLOTS
# ================================================================
DARK = "#0d1117"; GREY = "#161b22"; TEXT = "#e6edf3"
ACCENT = "#58a6ff"
plt.rcParams.update({
    "figure.facecolor": DARK, "axes.facecolor": GREY,
    "axes.edgecolor": "#30363d", "axes.labelcolor": TEXT,
    "xtick.color": TEXT, "ytick.color": TEXT,
    "text.color": TEXT, "grid.color": "#21262d",
    "font.family": "monospace",
})

# ── 1. Barras de métricas ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(DARK)
metrics_keys = ["accuracy","precision","recall","f1","roc_auc","avg_precision","mean_iou"]
labels_k     = ["Accuracy","Precision","Recall","F1","ROC-AUC","mAP","Mean IoU"]
vp = [m_p[k] if not np.isnan(m_p[k]) else 0 for k in metrics_keys]
vn = [m_n[k] if not np.isnan(m_n[k]) else 0 for k in metrics_keys]

x  = np.arange(len(labels_k))
w  = 0.35
ax = axes[0]
ax.bar(x - w/2, vp, w, label="PROFE", color=HEX_PROFE, alpha=0.9)
ax.bar(x + w/2, vn, w, label="NUEVO", color=HEX_NUEVO, alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(labels_k, rotation=30, ha="right", fontsize=8)
ax.set_ylim(0, 1.15); ax.set_title("Comparación de Métricas", fontsize=11, color=TEXT)
ax.legend(facecolor=GREY, edgecolor="#30363d"); ax.grid(axis="y", alpha=0.4)

# Etiqueta N/A si roc_auc es nan
for i, k in enumerate(metrics_keys):
    if k == "roc_auc":
        if np.isnan(m_p["roc_auc"]):
            ax.text(i - w/2, 0.05, "N/A", ha="center", fontsize=7, color=HEX_PROFE)
        if np.isnan(m_n["roc_auc"]):
            ax.text(i + w/2, 0.05, "N/A", ha="center", fontsize=7, color=HEX_NUEVO)

# Confusion matrices en axes[1]
ax2 = axes[1]; ax2.axis("off")
table_data = [["","Pred 0","Pred 1"],
              ["Act 0",f"TN={m_p['tn']}",f"FP={m_p['fp']}"],
              ["Act 1",f"FN={m_p['fn']}",f"TP={m_p['tp']}"]]
table2     = [["","Pred 0","Pred 1"],
              ["Act 0",f"TN={m_n['tn']}",f"FP={m_n['fp']}"],
              ["Act 1",f"FN={m_n['fn']}",f"TP={m_n['tp']}"]]
y_base = 0.85
for title, tdata, color in [
        ("PROFE – Confusion Matrix", table_data, HEX_PROFE),
        ("NUEVO – Confusion Matrix", table2,     HEX_NUEVO)]:
    ax2.text(0.02 if "PROFE" in title else 0.52, y_base+0.08,
             title, transform=ax2.transAxes, color=color, fontsize=9, fontweight="bold")
    for r, row in enumerate(tdata):
        for c, val in enumerate(row):
            xp = (0.02 if "PROFE" in title else 0.52) + c * 0.14
            yp = y_base - r * 0.22
            bg = "#1f2937" if r==0 or c==0 else (
                 "#134e2a" if (r==2 and c==2) else
                 "#7f1d1d" if (r==1 and c==2 or r==2 and c==1) else "#1f2937")
            ax2.text(xp, yp, val, transform=ax2.transAxes,
                     ha="center", va="center", fontsize=8, color=TEXT,
                     bbox=dict(facecolor=bg, edgecolor="#30363d", boxstyle="round,pad=0.3"))

plt.tight_layout()
bar_path = PLOT_DIR / "metricas_barras.png"
plt.savefig(bar_path, dpi=130, bbox_inches="tight", facecolor=DARK); plt.close()
print(f"  Saved: {bar_path}")

# ── 2. ROC Curve ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(DARK); ax.set_facecolor(GREY)
for (label, m, color) in [("PROFE", m_p, HEX_PROFE), ("NUEVO", m_n, HEX_NUEVO)]:
    if np.isnan(m["roc_auc"]):
        ax.text(0.5, 0.5 if label=="PROFE" else 0.4,
                f"{label}: ROC-AUC N/A\n(una sola clase en y_true)",
                ha="center", color=color, fontsize=9, transform=ax.transAxes)
        continue
    try:
        fpr, tpr, _ = roc_curve(m["y_true"], m["y_score"])
        ax.plot(fpr, tpr, color=color, lw=2,
                label=f"{label}  AUC={m['roc_auc']:.3f}")
        ax.fill_between(fpr, tpr, alpha=0.08, color=color)
    except Exception as e:
        print(f"  ROC plot error ({label}): {e}")
ax.plot([0,1],[0,1],"--",color="#30363d")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve", color=TEXT)
handles = ax.get_legend_handles_labels()
if handles[0]: ax.legend(facecolor=GREY, edgecolor="#30363d")
ax.grid(alpha=0.3)
roc_path = PLOT_DIR / "roc_curve.png"
plt.savefig(roc_path, dpi=130, bbox_inches="tight", facecolor=DARK); plt.close()
print(f"  Saved: {roc_path}")

# ── 3. Precision-Recall Curve ────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(DARK); ax.set_facecolor(GREY)
for (label, m, color) in [("PROFE", m_p, HEX_PROFE), ("NUEVO", m_n, HEX_NUEVO)]:
    if np.isnan(m["avg_precision"]):
        continue
    try:
        p, r, _ = precision_recall_curve(m["y_true"], m["y_score"])
        ax.plot(r, p, color=color, lw=2,
                label=f"{label}  AP={m['avg_precision']:.3f}")
        ax.fill_between(r, p, alpha=0.08, color=color)
    except Exception as e:
        print(f"  PR plot error ({label}): {e}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve", color=TEXT)
handles = ax.get_legend_handles_labels()
if handles[0]: ax.legend(facecolor=GREY, edgecolor="#30363d")
ax.grid(alpha=0.3)
pr_path = PLOT_DIR / "pr_curve.png"
plt.savefig(pr_path, dpi=130, bbox_inches="tight", facecolor=DARK); plt.close()
print(f"  Saved: {pr_path}")

# ── 4. IoU Distribution ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(DARK); ax.set_facecolor(GREY)
if m_p["iou_list"]:
    ax.hist(m_p["iou_list"], bins=30, color=HEX_PROFE, alpha=0.7,
            label=f"PROFE  μ={m_p['mean_iou']:.3f}")
if m_n["iou_list"]:
    ax.hist(m_n["iou_list"], bins=30, color=HEX_NUEVO, alpha=0.7,
            label=f"NUEVO  μ={m_n['mean_iou']:.3f}")
ax.axvline(IOU_THRESH, color="white", linestyle="--", alpha=0.6, label=f"IoU≥{IOU_THRESH}")
ax.set_xlabel("IoU"); ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de IoU (detecciones sobre GT)", color=TEXT)
ax.legend(facecolor=GREY, edgecolor="#30363d"); ax.grid(alpha=0.3)
iou_path = PLOT_DIR / "iou_distribution.png"
plt.savefig(iou_path, dpi=130, bbox_inches="tight", facecolor=DARK); plt.close()
print(f"  Saved: {iou_path}")

# ── 5. Inference time ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor(DARK); ax.set_facecolor(GREY)
ax.hist(stats["profe"]["inf_ms"], bins=40, color=HEX_PROFE, alpha=0.7,
        label=f"PROFE  μ={m_p['mean_inf_ms']:.1f}ms")
ax.hist(stats["nuevo"]["inf_ms"], bins=40, color=HEX_NUEVO, alpha=0.7,
        label=f"NUEVO  μ={m_n['mean_inf_ms']:.1f}ms")
ax.set_xlabel("Tiempo inferencia (ms)"); ax.set_ylabel("Frecuencia")
ax.set_title("Distribución de Tiempo de Inferencia", color=TEXT)
ax.legend(facecolor=GREY, edgecolor="#30363d"); ax.grid(alpha=0.3)
inf_path = PLOT_DIR / "inference_time.png"
plt.savefig(inf_path, dpi=130, bbox_inches="tight", facecolor=DARK); plt.close()
print(f"  Saved: {inf_path}")

# ================================================================
# REPORTE HTML
# ================================================================
def fmt_val(name, v):
    if np.isnan(v): return "N/A"
    if "ms" in name:    return f"{v:.1f} ms"
    if "conf" in name.lower(): return f"{v:.3f}"
    return f"{v*100:.2f}%"

winner = lambda vp, vn: ("🏆","—") if vp>vn else ("—","🏆") if vn>vp else ("—","—")

rows = [
    ("Accuracy",            m_p["accuracy"],       m_n["accuracy"],       True),
    ("Precision",           m_p["precision"],      m_n["precision"],      True),
    ("Recall",              m_p["recall"],         m_n["recall"],         True),
    ("F1-Score",            m_p["f1"],             m_n["f1"],             True),
    ("ROC-AUC",             m_p["roc_auc"],        m_n["roc_auc"],        True),
    ("Avg Precision (mAP)", m_p["avg_precision"],  m_n["avg_precision"],  True),
    ("Mean IoU",            m_p["mean_iou"],       m_n["mean_iou"],       True),
    ("Mean Confidence",     m_p["mean_conf"],      m_n["mean_conf"],      True),
    ("Inf. Time (ms)",      m_p["mean_inf_ms"],    m_n["mean_inf_ms"],    False),
]

table_rows_html = ""
for name, vp, vn, higher_better in rows:
    wp, wn = ("—","—")
    if not (np.isnan(vp) or np.isnan(vn)):
        wp, wn = winner(vp, vn) if higher_better else winner(-vp, -vn)
    vp_s = fmt_val(name, vp)
    vn_s = fmt_val(name, vn)
    table_rows_html += f"""
        <tr>
          <td>{name}</td>
          <td class="val cyan">{vp_s} {wp}</td>
          <td class="val green">{vn_s} {wn}</td>
        </tr>"""

cm_rows_html = ""
for label, m, color in [("PROFE", m_p, "cyan"), ("NUEVO", m_n, "green")]:
    cm_rows_html += f"""
    <div class="cm-block">
      <h3 class="{color}">{label} – Confusion Matrix</h3>
      <table class="cm-table">
        <tr><th></th><th>Pred 0</th><th>Pred 1</th></tr>
        <tr><td>Act 0</td><td class="tn">TN<br>{m['tn']}</td><td class="fp">FP<br>{m['fp']}</td></tr>
        <tr><td>Act 1</td><td class="fn">FN<br>{m['fn']}</td><td class="tp">TP<br>{m['tp']}</td></tr>
      </table>
    </div>"""

roc_note_p = f"<br><small>⚠ {m_p['roc_warn']}</small>" if m_p.get("roc_warn") else ""
roc_note_n = f"<br><small>⚠ {m_n['roc_warn']}</small>" if m_n.get("roc_warn") else ""

html_content = f"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<title>Reporte Analítico – Comparación de Modelos</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600;700&family=Space+Grotesk:wght@400;600;700&display=swap');
  *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
  :root {{
    --bg:#0d1117;--surface:#161b22;--border:#30363d;
    --text:#e6edf3;--muted:#8b949e;
    --cyan:#00ffff;--green:#00ff88;--blue:#58a6ff;
    --red:#ff6b6b;--yellow:#ffd700;
  }}
  body {{ background:var(--bg);color:var(--text);font-family:'Space Grotesk',sans-serif;padding:2rem; }}
  h1 {{ font-family:'JetBrains Mono',monospace;font-size:1.6rem;color:var(--cyan);margin-bottom:.3rem; }}
  .subtitle {{ color:var(--muted);font-size:.85rem;margin-bottom:2rem; }}
  .section {{ margin-bottom:2.5rem; }}
  h2 {{ font-family:'JetBrains Mono',monospace;font-size:1rem;color:var(--blue);border-bottom:1px solid var(--border);padding-bottom:.5rem;margin-bottom:1rem; }}
  h3 {{ font-size:.9rem;margin-bottom:.5rem; }}
  .cyan {{ color:var(--cyan); }} .green {{ color:var(--green); }}
  .metrics-table {{ width:100%;border-collapse:collapse;font-family:'JetBrains Mono',monospace;font-size:.82rem; }}
  .metrics-table th {{ background:var(--surface);color:var(--muted);padding:.5rem 1rem;text-align:left;border:1px solid var(--border); }}
  .metrics-table td {{ padding:.5rem 1rem;border:1px solid var(--border); }}
  .metrics-table tr:nth-child(even) td {{ background:#0f1419; }}
  .val {{ text-align:right;font-weight:600;font-size:.9rem; }}
  .metrics-table td.cyan {{ color:var(--cyan); }} .metrics-table td.green {{ color:var(--green); }}
  .cm-wrap {{ display:flex;gap:2rem;flex-wrap:wrap; }}
  .cm-block {{ background:var(--surface);border:1px solid var(--border);border-radius:8px;padding:1rem 1.5rem; }}
  .cm-table {{ border-collapse:collapse;font-family:'JetBrains Mono',monospace;font-size:.85rem;margin-top:.5rem; }}
  .cm-table th,.cm-table td {{ border:1px solid var(--border);padding:.5rem 1rem;text-align:center; }}
  .cm-table th {{ background:#21262d;color:var(--muted); }}
  .tp {{ background:#0d2818;color:#4ade80;font-weight:700; }}
  .tn {{ background:#1a2540;color:var(--blue);font-weight:700; }}
  .fp {{ background:#3b1d1d;color:var(--red);font-weight:700; }}
  .fn {{ background:#3b2600;color:var(--yellow);font-weight:700; }}
  .plots-grid {{ display:grid;grid-template-columns:repeat(auto-fit,minmax(420px,1fr));gap:1rem; }}
  .plot-card {{ background:var(--surface);border:1px solid var(--border);border-radius:8px;overflow:hidden; }}
  .plot-card img {{ width:100%;display:block; }}
  .plot-card p {{ padding:.4rem .8rem;font-size:.75rem;color:var(--muted); }}
  .cards {{ display:flex;gap:1rem;flex-wrap:wrap;margin-bottom:1.5rem; }}
  .card {{ background:var(--surface);border:1px solid var(--border);border-radius:8px;padding:1rem 1.5rem;min-width:160px; }}
  .card .label {{ font-size:.7rem;color:var(--muted);text-transform:uppercase;letter-spacing:.08em; }}
  .card .value {{ font-size:1.4rem;font-weight:700;font-family:'JetBrains Mono',monospace;margin-top:.2rem; }}
  .note {{ font-size:.75rem;color:var(--muted);font-style:italic; }}
  footer {{ margin-top:3rem;font-size:.75rem;color:var(--muted);border-top:1px solid var(--border);padding-top:1rem; }}
</style>
</head>
<body>
<h1>🎾 Model Analytics Report</h1>
<p class="subtitle">PROFE vs NUEVO · Ball Padel Detection · IoU threshold = {IOU_THRESH} · {len(image_paths)} imágenes evaluadas</p>

<div class="section">
  <h2>01 / Summary Scorecards</h2>
  <div class="cards">
    <div class="card"><div class="label">PROFE F1</div><div class="value cyan">{m_p['f1']*100:.1f}%</div></div>
    <div class="card"><div class="label">NUEVO F1</div><div class="value green">{m_n['f1']*100:.1f}%</div></div>
    <div class="card"><div class="label">PROFE ROC-AUC</div><div class="value cyan">{"N/A" if np.isnan(m_p['roc_auc']) else f"{m_p['roc_auc']:.3f}"}{roc_note_p}</div></div>
    <div class="card"><div class="label">NUEVO ROC-AUC</div><div class="value green">{"N/A" if np.isnan(m_n['roc_auc']) else f"{m_n['roc_auc']:.3f}"}{roc_note_n}</div></div>
    <div class="card"><div class="label">PROFE Mean IoU</div><div class="value cyan">{m_p['mean_iou']:.3f}</div></div>
    <div class="card"><div class="label">NUEVO Mean IoU</div><div class="value green">{m_n['mean_iou']:.3f}</div></div>
    <div class="card"><div class="label">PROFE Inf.</div><div class="value cyan">{m_p['mean_inf_ms']:.1f}ms</div></div>
    <div class="card"><div class="label">NUEVO Inf.</div><div class="value green">{m_n['mean_inf_ms']:.1f}ms</div></div>
  </div>
  <p class="note">⚠ ROC-AUC = N/A cuando el dataset de validación tiene solo una clase (e.g., todas las imágenes tienen pelota). Es un problema del split, no del modelo.</p>
</div>

<div class="section">
  <h2>02 / Metrics Table</h2>
  <table class="metrics-table">
    <thead><tr><th>Metric</th><th>PROFE (cyan)</th><th>NUEVO (green)</th></tr></thead>
    <tbody>{table_rows_html}</tbody>
  </table>
</div>

<div class="section">
  <h2>03 / Confusion Matrices</h2>
  <div class="cm-wrap">{cm_rows_html}</div>
</div>

<div class="section">
  <h2>04 / Plots</h2>
  <div class="plots-grid">
    <div class="plot-card"><img src="plots/metricas_barras.png"><p>Comparación general de métricas y matrices de confusión</p></div>
    <div class="plot-card"><img src="plots/roc_curve.png"><p>ROC Curve – area bajo la curva</p></div>
    <div class="plot-card"><img src="plots/pr_curve.png"><p>Precision-Recall Curve – Average Precision</p></div>
    <div class="plot-card"><img src="plots/iou_distribution.png"><p>Distribución de IoU sobre detecciones válidas</p></div>
    <div class="plot-card"><img src="plots/inference_time.png"><p>Distribución del tiempo de inferencia por imagen</p></div>
  </div>
</div>

<footer>
  Generado con Ultralytics YOLO · scikit-learn · matplotlib · Videos: {len(video_paths)} archivos en {VIDEOS_ROOT}
</footer>
</body>
</html>"""

with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(html_content)
print(f"\n✅ Reporte HTML: {OUTPUT_HTML}")

# ── Mostrar en notebook ──────────────────────────────────────
try:
    display(Video(str(OUTPUT_VIDEO), embed=True))
    display(HTML(f'<a href="{OUTPUT_HTML}" target="_blank">📊 Abrir reporte analítico completo</a>'))
except Exception:
    pass
print("\n✅ Todo listo. Revisa comparacion_modelos_ball/")